# Eukaryotic toehold switch — two-input AND

Manual test harness for `engine.gates.toehold.ToeholdAndGate`, built against a
**eukaryotic host** (`Host.HUMAN`/`Host.YEAST`, Kozak-cap-dependent-scanning
track — see `ToeholdGate`'s own docstring on why `host` is a constructor
parameter rather than a subclass). It inherits every method from `ToeholdGate`
except `generate_designs`, and sets `max_inputs = 2`. Scientific methods print
`pending Step 5` until the bodies land.

This notebook is `toehold_and.ipynb`'s sibling, not a replacement — that one
exercises the same stub against `Host.ECOLI` (steric RBS occlusion, serial
hairpins). Kept separate rather than parameterised over host in one notebook
because, once Step 5 lands, the two tracks are expected to diverge in real
construction (Kozak/scanning-blockage vs. RBS/steric-occlusion — see
`docs/modalities.md` and `ToeholdGate`'s own single-input eukaryotic work,
`toehold/eukaryotic-usage-example.ipynb`, for how much that single-input case
already had to diverge), not just in which fixed motif gets substituted in.

---

**Self-test variant.** From the "Fusing two ranked single-input designs" section
onward, this copy pairs the **sele** target against **itself** instead of against a
second gene (`areg`) — every downstream cell (`fuse`, `measure`, the Kozak/prefix
self-binding checks, the report) is otherwise unchanged. The point is a sanity check
free of any areg-vs-sele confound: every candidate below is genuinely two different
trigger windows drawn from one transcript, so its own report can additionally show
*where on that transcript* each of the report's triggers actually sits — see the
report's final section.

## The mechanism

An AND toehold opens only when **both** triggers are present. The
single-input eukaryotic case (`EukaryoticToeholdGate`, already built — see
`toehold/eukaryotic-usage-example.ipynb`) blocks a scanning 40S ribosome with a
closed hairpin sitting *upstream* of Kozak+AUG, not by sequestering Kozak
itself the way the prokaryotic RBS-in-loop layout sequesters the RBS. Extending
that to two inputs raises questions the prokaryotic serial-stem shape doesn't
have to answer, none of them resolved here — this notebook watches for them
once Step 5 lands, it doesn't decide them:

* **Where do two hairpins sit relative to one Kozak?** The prokaryotic AND
  puts the start codon inside the *inner* of two nested hairpins, with an RBS
  free in each hairpin's own loop. A `"trailing"`-style eukaryotic construct has
  no RBS-in-loop to nest a second one inside — the open question is whether the
  outer hairpin blocks scanning *before* it ever reaches an inner hairpin+Kozak,
  or whether Kozak itself gets sequestered by the inner stem the way the
  prokaryotic AUG does today.
* **Order still matters** — A-outer/B-inner is a different construct from the
  reverse, same as the prokaryotic case.
* **The intermediate (one-trigger) state is real** — if a scanning ribosome can
  already reach Kozak+AUG with only the first trigger present, the gate is an OR
  wearing an AND's shape. Evaluate the single-trigger states explicitly once
  `evaluate_design` exists for this class.
* **Kozak self-binding risk, doubled.** The single-input case already tracks
  whether a trigger-derived toehold happens to carry Kozak's reverse complement
  (`_kozak_rc_in_toehold`, and its worst-case `predicted_leakage` penalty in
  `evaluate_design` — eukaryotic-only, gated on `self.host.track`). An AND gate
  has two trigger-derived toeholds and one Kozak copy; whichever construction
  Step 5 picks needs the same check run against *both*, not just the one
  nearest Kozak in sequence.

## Setup

In [99]:
# Put the shared _fixtures.py on the path. It lives in the notebooks/ root, one
# level up from this gate's folder; search upward so the notebook works wherever
# Jupyter is launched. fx.bootstrap() then adds <repo>/src so `import engine...`
# resolves. No Django, no worker, no pipeline.
import sys, pathlib

for _base in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_base / '_fixtures.py').exists():
        if str(_base) not in sys.path:
            sys.path.insert(0, str(_base))
        break

import _fixtures as fx
fx.bootstrap()

engine package root: /Users/zeev/Projects/cernal_software/src


PosixPath('/Users/zeev/Projects/cernal_software/src')

## Fusing two ranked single-input designs into an AND candidate

`ToeholdAndGate.generate_designs` is still `NotImplementedError("Step 5")`, so nothing
below asks the gate to build anything. This section takes two **already-ranked
single-input eukaryotic toeholds** — one CSV per target — splices them into one serial
construct **in both orders**, measures each, and reports the best. It is a
notebook-level construction; when Step 5 lands, `generate_designs` owns this and this
section becomes the thing to check it against.

**The inputs** are NucSyn runner exports (`<target>/<profile>/results_ranked.csv`,
topology `euk_open_5p_kozak_after_stem`), already sorted best-first by
`postgen_global_rank`. Four columns carry the work: `switch_strand`, `trigger_seq`,
`switch_domain_breakdown_json` (named 0-indexed half-open domain bounds), and
`sim_score` plus friends for the producer's own verdict on each half.

**Why the domain breakdown and not string matching.** `kozak_seq` in these exports is
the degenerate IUPAC *pattern* (`GCMAMMAUG`, M = A or C), not the sequence that actually
landed in the construct, so `switch_strand.endswith(kozak_seq)` is always false. The
construct also does not end at the Kozak — 30 nt of reporter CDS (`exp_gene`) follow it.
The breakdown names every domain and its exact bounds, in the same 0-indexed half-open
convention this repo uses (`CLAUDE.md` §6), so the cut points are read rather than
guessed.

**The construction.** A design here reads

```
prefix | toehold | stem_up | loop | stem_down | pre_kozak | kozak | ext_kozak | opt_exp_gene | exp_gene
```

The **5' half** keeps everything before its `kozak` domain starts; the **3' half** keeps
everything after its `prefix` domain ends. Both hairpins therefore sit in series between
one prefix and one Kozak+AUG+reporter — the shape `process_trigger_p_and` builds in the
team's own `CERNAL_FUNCTIONS.py`.

**Both orders are built.** areg→sele and sele→areg are different molecules, not two
views of one: which hairpin a scanning ribosome meets first, and which trigger's toehold
sits at the exposed 5' end, both change. They are measured separately and ranked
together.

**What gets measured**, per construct, as raw kcal/mol:

| state | folded as | what it says |
|---|---|---|
| OFF | fused switch alone | both hairpins closed, nothing bound |
| +5' | `fused&trigger5` | the 5' half's trigger only — an intermediate state |
| +3' | `fused&trigger3` | the 3' half's trigger only — the other intermediate |
| +both | `fused&trigger5&trigger3` | the state the gate is supposed to need |

plus, on the OFF state, **the accessibility of both toeholds** — mean P(unpaired) over
each toehold span, from `FoldEngine.base_pair_probabilities` via the same
`_mean_unpaired` helper `ToeholdGate.evaluate_design` uses, so these numbers stay
comparable with the single-input ones. An AND built this way needs *both* toeholds open
in the OFF state: each trigger nucleates at its own, and neither can start on a toehold
that is already paired.

Trigger self-folding is **read from the CSV** (`energy_trigger_mfe`), not recomputed —
recomputing it with a slightly different model would produce two numbers for one
quantity (`CLAUDE.md` §5).

**Ranking is a notebook stand-in, not `engine.scoring`** (`CLAUDE.md` §3). The sort key
is one named constant in the run cell, and the ΔG sign convention is spelled out there —
"highest" is ambiguous for a quantity that is negative when it is good.

In [ ]:
# --- inputs -------------------------------------------------------------------
# Self-test variant of toehold_and_eu.ipynb: both halves come from the SAME target
# (sele) instead of pairing two different genes. Structurally identical pipeline --
# fuse()/measure()/the report are untouched -- the only real differences are here (one
# file instead of two) and in the run cell (excluding a design from pairing with
# itself, since that would reuse one trigger for both inputs of the "AND").
#
# NucSyn runner export, already sorted best-first by postgen_global_rank. This section
# preserves that order and never re-ranks within a file.
CSV_SELE = "/Users/zeev/Projects/NucSyn_Public_Platform_0.1.1/nucsyn-platform-work/output_T1/targets/sele/basic_switch/results_ranked.csv"

# One order: sele paired with itself. Unlike the two-target notebook, a second
# "sele -> sele" entry would be a literal duplicate of the same grid (both pools are
# the same list), not a different molecule -- so there is only one entry here, not
# two, and the nested loop over the same pool for both roles already reaches every
# ordered pair.
ORDERS = (("sele", CSV_SELE, "sele", CSV_SELE),)

# Below this many shared nt on the target transcript, two windows count as distinct
# trigger regions and are allowed to pair (a design is never paired with itself, but a
# few nt of incidental overlap between two otherwise-different windows is fine); at or
# above it, they are treated as the same underlying signal and excluded -- see the
# overlap check in the run cell below.
MIN_TRIGGER_SEPARATION_NT = 8

COLUMNS = {
    "switch": "switch_strand",
    "trigger": "trigger_seq",
    "domains": "switch_domain_breakdown_json",
    "rank": "postgen_global_rank",
    # Exact, correctly-sized span of the trigger on its target transcript --
    # trigger_end_pos - trigger_start_pos == len(trigger_seq), verified against real
    # rows (canonical_trigger_end_pos is wider and is not this). Used only by the
    # trigger-position map at the end of the report.
    "trigger_start": "trigger_start_pos",
    "trigger_end": "trigger_end_pos",
}

# The sele target's own full mRNA length -- from the runner YAML's target.sequence
# (zeev-single-eu-sele.yaml), cross-checked against this same CSV family's own scanned
# range (trigger_selection_full.csv: max toehold_end 2933, 0-indexed -> length 2934).
# A fixed, sourced constant rather than inferred from whichever rows TOP_5P/TOP_3P
# happens to load -- a partial sample could under-report the true length.
SELE_TRANSCRIPT_LENGTH = 2934

# The producing platform's own per-design numbers, carried through for context and
# never recomputed here (recomputing with a slightly different model would produce two
# numbers for one quantity). sim_score is its overall objective and is **lower-is-better**
# -- it is what postgen_global_rank is built from. The rest are the components the
# length trade-off actually shows up in: structure fidelity, toehold openness, and how
# much the trigger folds on itself.
CARRY_METRICS = (
    "sim_score",
    "mcc",
    "switch_toehold_unpaired_prob_mean",
    "energy_trigger_mfe",
)

# Where each half is cut, by domain name rather than by length.
CUT_5P_BEFORE = "kozak"  # the 5' half keeps everything before its kozak starts
KEEP_3P_AFTER = "prefix"  # the 3' half keeps everything after its prefix ends

# New architecture: a low-GC spacer physically separating the two hairpins, instead of
# splicing them directly together. Chosen by search + verified by real folding (not
# assumed): 0% GC, folds fully unpaired alone (MFE 0.0, all-dots structure) and shows no
# favourable interaction folded against a real trigger (MFE ~0.1, i.e. essentially none).
# Low GC keeps it unlikely to self-fold or to pair with anything else in the construct --
# A-U pairs are weaker than G-C, so a spacer built only from A/U has little to offer any
# competing structure even if a stray complementary stretch existed.
SPACER_SEQ = "AUUUUAUUUUUUUAUUAUUA"

# Grid size per order. Cost is ~0.22 s per construct (four folds plus one base-pair
# probability matrix), and the grid is TOP_5P x TOP_3P x 2 orders -- so 20 x 20 is
# ~800 constructs, ~3 minutes. 1200 x 1200 is not a thing to launch by accident.
TOP_5P = 20
TOP_3P = 20

TEMPERATURE_C = 37.0  # recorded on every energy below; see FoldEngine's docstring
REPORT_TOP_N = 15  # how many combined designs the report details
# Reports are named by what they are, so the AND and single-switch outputs never collide.
AND_STEM = f"{ORDERS[0][0]}_{ORDERS[0][2]}"

SHOW_BPP = True  # base-pair probability matrices under the structure plots


In [101]:
import csv
import json
from pathlib import Path

from engine.gates.toehold import _mean_unpaired  # same accessibility helper measure() uses


def _maybe_float(value):
    """``float(value)``, or ``None`` when the cell is blank or not a number.

    Deliberately not ``float(value or 0)``: a blank cell is a measurement that is not
    there, and 0.0 is a real value that would go on to sort as though it had been
    measured (``CLAUDE.md`` §3).
    """
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def motif_self_bind_score(sequence, motif_seq):
    """Best fractional match, sliding a window the length of ``motif_seq`` across
    ``sequence``, against the reverse complement of ``motif_seq`` -- built entirely from
    existing engine functions (``sq.windows`` for the scan, ``sq.reverse_complement`` for
    the target, ``sq.hamming`` to score each window; matches = window length minus
    mismatches), not a hand-rolled loop over either.

    Generic over which fixed motif is being checked -- used below for both Kozak
    (``own_kozak_rc_score``, ``kozak_rc_5p``) and the prefix (``own_prefix_rc_score``,
    ``prefix_rc_3p``): the risk is the same shape either way, a fixed element that could
    fold back onto a toehold regardless of how far away it sits.

    Same sliding-window partial-match idea as the team's own ``complementarity_Kozak`` in
    ``CERNAL_FUNCTIONS.py`` — and the one ``engine.gates.toehold._kozak_rc_in_toehold``
    should arguably also use: an exact full-length substring check misses a real,
    ViennaRNA-confirmed partial match (a 5/9 toehold x Kozak+AUG pair found directly in a
    real MFE fold earlier this session, at a window score of 0.556 — see that function's
    own docstring for the caveat this generalises past). Score in [0, 1]; 1.0 is a perfect
    match over the whole window.
    """
    target = fx.sq.reverse_complement(motif_seq)
    window_len = len(target)
    if len(sequence) < window_len:
        return 0.0
    return max(
        1.0 - fx.sq.hamming(window, target) / window_len
        for _start, window in fx.sq.windows(sequence, window_len)
    )


def _toehold_self_fold(toehold_seq, folder):
    """Fraction of ``toehold_seq`` paired to itself when folded alone.

    A general self-structure tendency, independent of what it might specifically bind to
    (Kozak or otherwise) — read alongside ``kozak_self_bind_score``, not instead of it: a
    toehold can fold into a hairpin that has nothing to do with Kozak and still be
    unusable, and this is the only one of the two checks that would catch that.
    """
    matrix = folder.base_pair_probabilities(toehold_seq)
    return 1.0 - _mean_unpaired(matrix, 0, len(toehold_seq))


def load_ranked_designs(path, folder, columns=COLUMNS, limit=None):
    """Rows of a NucSyn ``results_ranked.csv``, kept in file order (already ranked).

    Returns ``(rows, problems)``. A row missing a column, carrying a non-RNA sequence, or
    whose domain breakdown does not tile its own switch exactly is collected in
    ``problems`` with its line number and reason — never dropped silently or patched to a
    default.

    Sequences are normalised with ``engine.sequences.to_rna`` (re-exported as ``fx.sq``)
    here at the edge: a CSV is exactly where a DNA alphabet sneaks in, and ViennaRNA
    reads a T as an unknown base rather than failing (``CLAUDE.md`` §6).

    Three metrics are *computed* here rather than read from the CSV, per design as it
    stands before any fusion decision: ``own_kozak_rc_score`` and ``own_prefix_rc_score``
    (this design's own toehold against its own, still-present Kozak and prefix
    respectively) and ``toehold_self_fold`` (the toehold's general self-pairing tendency,
    folded alone). All three land in ``metrics`` alongside the CSV-carried ones.
    ``own_kozak_rc_score`` matters less for whichever half ends up 5' in a fused pair --
    its own Kozak gets cut away at fusion (see ``fuse()``) -- and symmetrically
    ``own_prefix_rc_score`` matters less for whichever half ends up 3' -- its own prefix
    is what gets cut. Both are computed for every row regardless, since which role a
    design plays is decided later, per pair, not at load time.
    """
    rows, problems = [], []
    with Path(path).open(newline="") as handle:
        for line_no, raw in enumerate(csv.DictReader(handle), start=2):  # 2 = first data line
            missing = [column for column in columns.values() if not (raw.get(column) or "").strip()]
            if missing:
                problems.append((line_no, f"missing or empty: {', '.join(missing)}"))
                continue

            switch = fx.sq.to_rna(raw[columns["switch"]])
            trigger = fx.sq.to_rna(raw[columns["trigger"]])
            not_rna = [
                name
                for name, value in (("switch", switch), ("trigger", trigger))
                if not fx.sq.is_valid_rna(value)
            ]
            if not_rna:
                problems.append((line_no, f"not RNA after to_rna(): {', '.join(not_rna)}"))
                continue

            # 0-indexed, start-inclusive/end-exclusive, same convention as the engine.
            domains = {
                domain["name"]: (domain["start"], domain["end"])
                for domain in json.loads(raw[columns["domains"]])
            }
            # The breakdown is the only thing locating the cut points, so a breakdown that
            # does not tile the switch exactly would put those cuts somewhere arbitrary.
            covered = sorted(domains.values())
            if not covered or covered[0][0] != 0 or covered[-1][1] != len(switch):
                problems.append((line_no, "domain breakdown does not span the switch"))
                continue

            toehold_seq = switch[domains["toehold"][0] : domains["toehold"][1]]
            own_kozak_span = domains.get("kozak")
            own_kozak_seq = (
                switch[own_kozak_span[0] : own_kozak_span[1]] if own_kozak_span else None
            )
            own_prefix_span = domains.get("prefix")
            own_prefix_seq = (
                switch[own_prefix_span[0] : own_prefix_span[1]] if own_prefix_span else None
            )
            metrics = {name: _maybe_float(raw.get(name)) for name in CARRY_METRICS}
            metrics["own_kozak_rc_score"] = (
                motif_self_bind_score(toehold_seq, own_kozak_seq) if own_kozak_seq else None
            )
            metrics["own_prefix_rc_score"] = (
                motif_self_bind_score(toehold_seq, own_prefix_seq) if own_prefix_seq else None
            )
            metrics["toehold_self_fold"] = _toehold_self_fold(toehold_seq, folder)

            # Optional: only this notebook's COLUMNS declares trigger_start/trigger_end
            # (the single-input notebook's loader has neither, and doesn't need them --
            # this is purely for the trigger-position map at the end of this report).
            trigger_start = int(raw[columns["trigger_start"]]) if "trigger_start" in columns else None
            trigger_end = int(raw[columns["trigger_end"]]) if "trigger_end" in columns else None

            rows.append(
                {
                    "switch": switch,
                    "trigger": trigger,
                    "domains": domains,
                    "rank": raw[columns["rank"]],
                    "metrics": metrics,
                    "line_no": line_no,
                    "trigger_start": trigger_start,
                    "trigger_end": trigger_end,
                }
            )
            if limit is not None and len(rows) >= limit:
                break
    return rows, problems

In [102]:
import base64
from io import BytesIO

import matplotlib.pyplot as plt

# Three semantic outlines only, same convention as the single-input report's structure
# panels: no per-base A/U/G/C colour, since base identity is already in the sequence
# listing and colouring it there carries no information the reader needs here. Trigger A
# (5' half) and trigger B (3' half) get distinct colours throughout -- the OFF/ON
# structure plots draw both toeholds/both trigger strands in one picture, and without this
# there was no way to tell which outline belonged to which trigger.
_BINDING_COLOR_5P = "#4C6E8C"  # trigger A / 5' half -- toehold+stem footprint + trigger strand
_BINDING_COLOR_3P = "#B5507E"  # trigger B / 3' half -- same, distinct hue
_KOZAK_COLOR = "#2E9E56"  # Kozak + start codon

# Domain -> layout-bar colour, matching the single-input report's own palette for the
# domains the two chemistries share (toehold, stem, loop, Kozak) and extending it for the
# ones only this export names (prefix, the reporter CDS).
_DOMAIN_COLOR = {
    "prefix": "#DCE4D8",
    "toehold": "#4C6E8C",
    "stem_up": "#7E97AC",
    "loop": "#C7D0CA",
    "stem_down": "#7E97AC",
    "kozak": "#2E9E56",
    "exp_gene": "#B5760E",
}
# These four domain names appear on both halves -- disambiguated with a prime in the bar.
_PER_HALF_DOMAINS = {"toehold", "stem_up", "loop", "stem_down"}


def _png_from_fig(fig):
    buffer = BytesIO()
    fig.savefig(buffer, format="png", facecolor="white")
    plt.close(fig)
    buffer.seek(0)
    return base64.b64encode(buffer.read()).decode("ascii")


def _structure_png(sequence, structure, title, energy=None, spans=(), figsize=(3.6, 3.2), *, folder):
    """One folded structure as a base64 PNG, with `spans` outlined.

    Coordinates come from ``FoldEngine.layout_coordinates`` (the shared 2D layout: one
    point per base), so this notebook never imports ViennaRNA itself.
    """
    points = folder.layout_coordinates(structure)
    xs = [x for x, _ in points]
    ys = [y for _, y in points]

    fig, ax = plt.subplots(figsize=figsize, dpi=130)
    ax.plot(xs, ys, "-", color="#B9C2BC", linewidth=1.1, zorder=1)

    stack = []
    for i, character in enumerate(structure):
        if character == "(":
            stack.append(i)
        elif character == ")":
            j = stack.pop()
            ax.plot([xs[i], xs[j]], [ys[i], ys[j]], "-", color="#8899AA",
                    linewidth=0.9, zorder=1, alpha=0.7)

    for i in range(len(sequence)):
        ring = next((colour for start, end, colour in spans if start <= i < end), None)
        ax.scatter(xs[i], ys[i], s=24 if ring else 13, color="#C7D0CA",
                   edgecolors=ring or "#666666", linewidths=1.3 if ring else 0.3,
                   zorder=3 if ring else 2)

    ax.text(xs[0], ys[0] + 8, "5'", fontsize=7, ha="center", color="#345")
    ax.text(xs[-1], ys[-1] + 8, "3'", fontsize=7, ha="center", color="#345")
    ax.set_title(title if energy is None else f"{title} ({energy:.1f} kcal/mol)",
                 fontsize=9, fontweight="bold")
    ax.set_aspect("equal")
    ax.axis("off")
    plt.tight_layout()
    return _png_from_fig(fig)


def render_layout_bar_png(built, title="Switch layout — both hairpins in series", label_halves=True):
    """Segmented layout bar for a two-hairpin AND construct — modeled on the reference
    report's "Switch layout" figure, one labeled segment per surviving domain of *both*
    halves, positioned by its own fused-frame span from ``fuse()``."""
    segments = []
    for half_label, domains in (("5'", built["domains_5p"]), ("3'", built["domains_3p"])):
        for name, (start, end) in domains.items():
            length = end - start
            if length <= 0:  # this export's zero-length domains (pre_kozak, etc.)
                continue
            label = f"{name} {half_label}" if label_halves and name in _PER_HALF_DOMAINS else name
            segments.append((start, label, length, _DOMAIN_COLOR.get(name, "#B7C4BC")))
    if built.get("spacer_span"):
        s0, s1 = built["spacer_span"]
        if s1 > s0:
            segments.append((s0, "spacer", s1 - s0, "#9AA3AE"))
    segments.sort(key=lambda s: s[0])  # domain dict order need not match fused order

    total = len(built["fused"])
    fig, ax = plt.subplots(figsize=(9.6, 1.6), dpi=130)
    for start, label, length, color in segments:
        ax.barh(0, length, left=start, height=0.6, color=color, edgecolor="white", linewidth=1.2)
        if length / total > 0.02:
            ax.text(start + length / 2, 0, str(length), ha="center", va="center",
                    fontsize=7.5, fontweight="bold", color="white")
            ax.text(start + length / 2, -0.55, label, ha="center", va="top",
                    fontsize=6.6, color="#46524B")
    ax.set_xlim(0, total)
    ax.set_ylim(-1.15, 0.5)
    ax.axis("off")
    ax.set_title(title, fontsize=10, fontweight="bold", loc="left")
    plt.tight_layout()
    return _png_from_fig(fig)


def render_register_png(switch, footprint_span, trigger_sequence, title, color, kozak_span=None):
    """Switch (5'->3') vs. the trigger it was reverse-complemented from (3'->5'), aligned
    base-for-base under its own footprint — modeled on the reference report's "Designed
    antiparallel register" figure, generalised to a footprint that need not start at
    position 0 (the 3' half's footprint sits mid-construct here, not at the 5' end).
    ``color`` should match whichever trigger (A/5' or B/3') this register is for, so the
    two figures read as one pair with the OFF/ON structure plots."""
    start, end = footprint_span
    footprint_len = end - start
    # binding_region in generate_designs pairs with the trigger's *last* footprint_len nt
    # (see the single-input notebook's own register function) — the same holds here: each
    # half's toehold+stem_up is the reverse complement of its own trigger's tail.
    trigger_tail_reversed = trigger_sequence[-footprint_len:][::-1]

    n = len(switch)
    fig, ax = plt.subplots(figsize=(max(6.4, n * 0.1), 2.4), dpi=130)
    for i, base in enumerate(switch):
        ax.text(i, 1, base, ha="center", va="center", fontsize=6.3, family="monospace", fontweight="bold")
    for j, base in enumerate(trigger_tail_reversed):
        i = start + j
        ax.text(i, -1, base, ha="center", va="center", fontsize=6.3, family="monospace", color="#46524B")
        ax.plot([i, i], [0.7, -0.7], color=color, linewidth=0.9, zorder=0)

    if kozak_span is not None:
        k0, k1 = kozak_span
        ax.plot([k0, k1 - 1], [1.4, 1.4], color=_KOZAK_COLOR, linewidth=2)
        ax.text((k0 + k1 - 1) / 2, 1.6, "Kozak", ha="center", fontsize=6.5, color=_KOZAK_COLOR, fontweight="bold")

    ax.text(-1.5, 1, "5'", ha="right", va="center", fontsize=6.5, color="#345")
    ax.text(n, 1, "3'", ha="left", va="center", fontsize=6.5, color="#345")
    ax.text(start - 1.5, -1, "3'", ha="right", va="center", fontsize=6.5, color="#345")
    ax.text(end - 1 + 1.5, -1, "5'", ha="left", va="center", fontsize=6.5, color="#345")

    ax.set_xlim(-3, n + 3)
    ax.set_ylim(-2, 2.1)
    ax.axis("off")
    ax.set_title(title, fontsize=9, fontweight="bold", loc="left")
    plt.tight_layout()
    return _png_from_fig(fig)


def render_trigger_position_png(points, transcript_length, title, legend=()):
    """Where a set of triggers sits on the target transcript.

    ``points`` is an iterable of ``(position, colour, label)`` -- built by the caller,
    so this function serves both the AND lists (two coloured roles per pair) and the
    single-switch list (one colour) without knowing either row shape. It is
    de-duplicated here, not by the caller: the top-ranked pool is small enough that the
    same design is reused across many reported candidates, so plotting every occurrence
    piles identical ticks on one position, which is not "where are the triggers", just
    noise. ``legend`` is an iterable of ``(colour, label)`` drawn as proxy lines; pass
    () for a single-colour map that needs no key.

    Ticks are labelled with the design's own rank -- the same identifier the summary
    tables and candidate cards use -- so a mark here can be looked up against a specific
    row, not just a coordinate. The bottom axis carries the real genomic scale. A design
    that plays *both* roles across different candidates gets two ticks at its one
    position, one per colour: a real fact about the pool, not a duplicate to hide.
    """
    points = sorted(set(points), key=lambda p: p[0])
    if not points:
        return None

    # Greedy tier assignment: place each tick as low as it can go without its label
    # landing within min_gap of the last label already placed in that tier.
    min_gap = transcript_length * 0.012
    tier_last_position = {}
    placements = []
    for position, color, label in points:
        tier = 0
        while tier in tier_last_position and position - tier_last_position[tier] < min_gap:
            tier += 1
        tier_last_position[tier] = position
        placements.append((position, color, label, tier))

    max_tier = max(tier for _, _, _, tier in placements)
    tick_height, tier_gap = 0.5, 0.55

    fig, ax = plt.subplots(
        figsize=(max(10, transcript_length / 220), 1.5 + 0.4 * max_tier), dpi=130
    )
    ax.plot([0, transcript_length], [0, 0], color="#B9C2BC", linewidth=1.5, zorder=1)

    for position, color, label, tier in placements:
        top = tick_height + tier * tier_gap
        ax.plot([position, position], [0, top], color=color, linewidth=1.1, zorder=2)
        ax.text(
            position, top + 0.05, label, rotation=90, ha="center", va="bottom",
            fontsize=6, color=color, family="monospace",
        )

    for _legend_color, _legend_label in legend:
        ax.plot([], [], color=_legend_color, linewidth=2, label=_legend_label)
    if legend:
        ax.legend(loc="upper right", fontsize=7.5, frameon=False)

    for xt in range(0, transcript_length + 1, 500):
        ax.plot([xt, xt], [-0.15, 0], color="#C7D0CA", linewidth=0.8, zorder=1)
        ax.text(xt, -0.25, str(xt), ha="center", va="top", fontsize=6.5, color="#75837B")
    ax.text(
        transcript_length / 2, -0.6, "position on sele mRNA (nt)",
        ha="center", va="top", fontsize=7.5, color="#46524B",
    )

    ax.set_xlim(-transcript_length * 0.01, transcript_length * 1.01)
    ax.set_ylim(-0.9, tick_height + max_tier * tier_gap + 0.55)
    ax.axis("off")
    # fig-level suptitle, not ax.set_title: the legend already sits just inside the axes'
    # top-right corner, and an in-axes title at the same height would collide with it.
    fig.suptitle(title, fontsize=10, fontweight="bold", x=0.01, ha="left")
    plt.tight_layout()
    return _png_from_fig(fig)


print("report helpers defined")

report helpers defined


In [103]:
import html as _html
from datetime import datetime

# Same palette and print rules as the single-input report, so the two read as one family
# (var/reports/switch_candidate_report_areg.html) -- plus the two image classes that
# report's layout bar and register figures use.
_AND_CSS = """
  :root {
    --scr-ground: #F6F8F4; --scr-surface: #FFFFFF; --scr-surface-2: #EEF2EA;
    --scr-ink: #131A16; --scr-ink-2: #46524B; --scr-ink-3: #75837B; --scr-line: #DCE4D8;
    --scr-accent: #2E9E56; --scr-accent-glow: #E4F7EA; --scr-accent-ink: #146134;
    --scr-structural: #4C6E8C; --scr-structural-bg: #E9EFF4;
    --scr-amber: #B5760E; --scr-amber-bg: #FBF0DD;
    --scr-shadow: 0 1px 2px rgba(19,26,22,0.04), 0 8px 24px -12px rgba(19,26,22,0.12);
  }
  @media (prefers-color-scheme: dark) {
    :root:not([data-theme="light"]) {
      --scr-ground: #0E1512; --scr-surface: #141D18; --scr-surface-2: #1A2620;
      --scr-ink: #E9F2EC; --scr-ink-2: #AEC0B6; --scr-ink-3: #7C8D83; --scr-line: #26362D;
      --scr-accent: #4CE07A; --scr-accent-glow: #163524; --scr-accent-ink: #7DF0A2;
      --scr-structural: #8FB3D6; --scr-structural-bg: #1B2A38;
      --scr-amber: #E0AA4C; --scr-amber-bg: #362B14;
      --scr-shadow: 0 1px 2px rgba(0,0,0,0.3), 0 8px 24px -12px rgba(0,0,0,0.5);
    }
  }
  * { box-sizing: border-box; }
  body.scr { margin: 0; background: var(--scr-ground); color: var(--scr-ink);
    font-family: "IBM Plex Sans", system-ui, sans-serif; padding: 0 20px; }
  .scr-mono { font-family: "IBM Plex Mono", ui-monospace, monospace; font-variant-numeric: tabular-nums; }
  .scr-wrap { max-width: 980px; margin: 0 auto; padding-block: 44px 72px; }
  .scr-wrap h1 { font-size: clamp(26px, 4vw, 34px); font-weight: 700; line-height: 1.15;
    letter-spacing: -0.01em; margin: 0 0 8px; text-wrap: balance; }
  .scr-eyebrow { font-family: "IBM Plex Mono", monospace; font-size: 12px; letter-spacing: 0.08em;
    text-transform: uppercase; color: var(--scr-structural); margin: 0 0 10px; }
  .scr-dek { font-size: 15px; line-height: 1.6; color: var(--scr-ink-2); max-width: 72ch; margin: 0 0 32px; }
  .scr-wrap section { margin-bottom: 40px; }
  .scr-wrap h2 { font-size: 20px; font-weight: 700; letter-spacing: -0.01em; color: var(--scr-ink);
    margin: 0 0 12px; padding-bottom: 8px; border-bottom: 2px solid var(--scr-line); }
  .scr-wrap h3 { font-family: "IBM Plex Mono", monospace; font-size: 14px; font-weight: 600;
    color: var(--scr-accent-ink); background: var(--scr-accent-glow); display: inline-block;
    padding: 4px 10px; border-radius: 6px; margin: 0 0 14px; }
  .scr-p { font-size: 14px; line-height: 1.7; color: var(--scr-ink-2); margin: 0 0 12px; max-width: 78ch; }
  .scr-callout { background: var(--scr-structural-bg); border-left: 3px solid var(--scr-structural);
    border-radius: 4px; padding: 12px 16px; font-size: 13px; line-height: 1.6;
    color: var(--scr-ink-2); margin: 0 0 12px; }
  .scr-table-wrap { overflow-x: auto; border: 1px solid var(--scr-line); border-radius: 10px;
    background: var(--scr-surface); box-shadow: var(--scr-shadow); margin-bottom: 8px; }
  .scr-table-wrap table { width: 100%; border-collapse: collapse; font-size: 12.5px; }
  .scr-table-summary table { min-width: 900px; }
  .scr-table-wrap th { text-align: left; font-size: 10.5px; letter-spacing: 0.05em; text-transform: uppercase;
    color: var(--scr-ink-3); font-weight: 600; padding: 10px 12px; border-bottom: 1px solid var(--scr-line);
    background: var(--scr-surface-2); }
  .scr-table-wrap td { padding: 8px 12px; border-bottom: 1px solid var(--scr-line); color: var(--scr-ink-2); }
  .scr-table-wrap tr:last-child td { border-bottom: none; }
  .scr-table-wrap td.num { text-align: right; font-family: "IBM Plex Mono", monospace; }
  .scr-pill { display: inline-block; font-family: "IBM Plex Mono", monospace; font-size: 10px;
    font-weight: 600; letter-spacing: 0.03em; padding: 2px 6px; border-radius: 999px;
    background: var(--scr-structural-bg); color: var(--scr-structural); }
  .scr-candidate { background: var(--scr-surface); border: 1px solid var(--scr-line); border-radius: 12px;
    padding: 20px 22px; box-shadow: var(--scr-shadow); margin-bottom: 18px; }
  .scr-candidate-head { display: flex; align-items: baseline; justify-content: space-between;
    gap: 12px; margin-bottom: 12px; flex-wrap: wrap; }
  .scr-candidate-head h3 { margin: 0; }
  .scr-candidate-id { font-family: "IBM Plex Mono", monospace; font-size: 12px; color: var(--scr-ink-3); }
  .scr-layout-img { max-width: 100%; height: auto; margin-bottom: 14px; }
  .scr-register-img { max-width: 100%; height: auto; margin-top: 10px; }
  .scr-subgrid { display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));
    gap: 14px; margin-bottom: 16px; }
  .scr-subtable-title { font-size: 11px; font-weight: 600; letter-spacing: 0.04em; text-transform: uppercase;
    color: var(--scr-ink-3); margin: 0 0 6px; }
  .scr-seq-block { font-size: 11.5px; line-height: 1.8; margin-bottom: 4px; }
  .scr-seq-block .label { display: inline-block; width: 104px; color: var(--scr-ink-3);
    font-family: "IBM Plex Mono", monospace; }
  .scr-seq-block .seq { color: var(--scr-ink); word-break: break-all; }
  .scr-structure-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(260px, 1fr));
    gap: 14px; margin-top: 10px; }
  .scr-structure-grid figure { margin: 0; background: var(--scr-surface-2); border-radius: 8px;
    padding: 6px; text-align: center; }
  .scr-structure-grid img { width: 100%; height: auto; }
  .scr-structure-grid figcaption { font-size: 11px; color: var(--scr-ink-3); margin-top: 4px; }
  footer.scr-footer { border-top: 1px solid var(--scr-line); padding-top: 18px; font-size: 11.5px;
    color: var(--scr-ink-3); }
  @media print {
    :root, :root:not([data-theme="light"]) {
      --scr-ground: #FFFFFF; --scr-surface: #FFFFFF; --scr-surface-2: #F3F5F1;
      --scr-ink: #131A16; --scr-ink-2: #3A443E; --scr-ink-3: #5C6660; --scr-line: #C9D2C4;
      --scr-accent: #1F7A42; --scr-accent-glow: #E4F7EA; --scr-accent-ink: #0F4A26;
      --scr-structural: #35526B; --scr-structural-bg: #E9EFF4;
      --scr-amber: #8A5A0A; --scr-amber-bg: #FBF0DD; --scr-shadow: none;
    }
    body.scr { padding: 0 12px; }
    .scr-candidate, .scr-table-wrap { box-shadow: none; }
    .scr-candidate-head, .scr-subgrid > *, .scr-seq-block, .scr-structure-grid figure, tr {
      break-inside: avoid; page-break-inside: avoid;
    }
    .scr-candidate { break-inside: auto; page-break-inside: auto; }
    thead { display: table-header-group; }
  }
"""


def _table(rows):
    body = "".join(
        f'<tr><td>{_html.escape(str(k))}</td><td class="num">{_html.escape(str(v))}</td></tr>'
        for k, v in rows
    )
    return f'<div class="scr-table-wrap"><table><tbody>{body}</tbody></table></div>'


def _num(value, places=2):
    """'—' for a measurement that is not there, so a gap never reads as a zero."""
    return "—" if value is None else f"{value:.{places}f}"


# Part 1 — the single switch (no AND)

One gene, one trigger, one hairpin: the plain eukaryotic toehold switch straight from the
gene's ranked NucSyn export, measured as OFF vs. **+trigger**. Nothing in this part calls
`fuse()` or `measure()` — those belong to Part 2, which fuses two of these single switches
into an AND. Output: `var/reports/single_toehold_report_<gene>.html/.pdf`.

In [104]:
# ==================================================================================
# PART 1 -- SINGLE SWITCH: measure each ranked single design as OFF vs +trigger.
# ==================================================================================
import time

# One FoldEngine for the whole notebook (Part 2 reuses it): one cache, one temperature.
folder = fx.fold_engine(real=True, temperature=TEMPERATURE_C)

# Publish that one engine to the parallel-worker module, which folds through it instead
# of importing ViennaRNA itself (CLAUDE.md sec 1/sec 5: FoldEngine is the only place a
# gate family folds, and exactly one instance exists per run). The pools fork, so each
# worker inherits THIS instance -- set it here, where `folder` is born and before any
# pool can start, so every later sweep cell is covered by having run this cell at all.
import sys as _sys_eng
from pathlib import Path as _Path_eng

for _base in (_Path_eng.cwd(), *_Path_eng.cwd().parents):
    if (_base / "_joint_state_parallel.py").exists():
        if str(_base) not in _sys_eng.path:
            _sys_eng.path.insert(0, str(_base))
        break
from _joint_state_parallel import use_fold_engine

use_fold_engine(folder)
print(f"FoldEngine published to the worker module: {folder.versions()}")


def measure_single_switch(design, folder):
    """Raw measurements for one un-fused switch: the trigger-bound state vs. OFF.

    Same quantities ``measure()`` reports per AND state, for one trigger -- ``d_ee`` is the
    free-energy stabilisation the trigger buys (negative is better), ``access`` the OFF-state
    openness of the toehold, and the two ``*_rc`` values are the toehold's self-binding risk
    against the Kozak / prefix already computed at load time. The trigger goes in through
    ``&`` (a real two-molecule complex), never string concatenation.
    """
    switch, trigger = design["switch"], design["trigger"]
    bound = f"{switch}&{trigger}"
    out = {
        "mfe_off": folder.mfe(switch).energy,
        "mfe_on": folder.mfe(bound).energy,
        "ee_off": folder.partition(switch),
        "ee_on": folder.partition(bound),
    }
    out["d_mfe"] = out["mfe_on"] - out["mfe_off"]
    out["d_ee"] = out["ee_on"] - out["ee_off"]
    out["access"] = _mean_unpaired(folder.base_pair_probabilities(switch), *design["domains"]["toehold"])
    out["kozak_rc"] = design["metrics"]["own_kozak_rc_score"]
    out["prefix_rc"] = design["metrics"]["own_prefix_rc_score"]
    return out


# The quick single-CSV preview that used to run here (SINGLE_POOL/SINGLE_CSV/
# SINGLE_GENE, dG-only, no NUPACK) was removed -- dead weight, never read by anything
# below: the real pipeline is the top-150 three-CSV comparison in the next cell, which
# builds its own pool straight from THREE_CSVS/THREE_CSV_LABELS. `folder` and
# `measure_single_switch()` defined above are what that cell (and the final report's
# images cell) actually reuse from here.


In [105]:
import sys as _sys_p1cmp
from pathlib import Path as _Path_p1cmp

import nupack

_here_p1cmp = _Path_p1cmp.cwd()
for _base in (_here_p1cmp, *_here_p1cmp.parents):
    if (_base / "_joint_state_parallel.py").exists():
        if str(_base) not in _sys_p1cmp.path:
            _sys_p1cmp.path.insert(0, str(_base))
        break
from _joint_state_parallel import SingleStateJob, pick_fake_design, run_parallel_single

# ==================================================================================
# PART 1 -- SINGLE SWITCH: three-CSV comparison. Same top-N pool, same
# measure_single_switch()/load_ranked_designs() (dG ranking) and the same
# SingleStateJob/run_parallel_single (ViennaRNA real P(bound)) and NUPACK tube+MFE
# pattern as the cells above -- nothing here is a second implementation, only the same
# logic looped over three source exports instead of one.
#
# Per this task's own instruction: CSV_SELE stays pointed at the current default
# export (15/18/21 nt); the other two get their own explicit local path variables
# rather than repeatedly reassigning CSV_SELE.
# ==================================================================================
import hashlib
import time as _time3

CSV_SELE_OLD = "/Users/zeev/Projects/TOEHOLDS/sele_old/basic_switch/results_ranked.csv"  # toehold 9/15 nt, oldest export
CSV_SELE_CURRENT = CSV_SELE  # toehold 15/18/21 nt -- current default, unchanged
CSV_SELE_V1 = "/Users/zeev/Projects/TOEHOLDS/sele/basic_switch/results_ranked_v1.csv"  # toehold 24/27/30 nt, longest export

# Restricted to the current sele export only -- no cross-CSV comparison this run
# (CSV_SELE_OLD / CSV_SELE_V1 left defined above for later use, just not looped over).
# Both tuples stay the same shape (label, path) so every downstream cell's
# "for _label in THREE_CSV_LABELS" loop and zip(THREE_CSV_LABELS, THREE_CSVS) still
# work unchanged with one entry instead of three.
def _toehold_length_label(path, gene_name):
    """CSV label built from THIS FILE's own toehold domain lengths (scanned out of
    switch_domain_breakdown_json, never hand-typed -- see this function's own git
    history for the hand-typed label it replaced) PLUS an 8-hex-char content hash of
    the file's bytes.

    The toehold-length part alone is not unique: two different exports built from the
    same design_grid.toehold_sizes would collide on it, and every downstream dict this
    label keys (single_scored_by_csv, the single-switch disk cache, ...) would then
    silently let the second CSV's entry overwrite the first's, or -- worse, across
    separate notebook runs -- let a NEW file with a coincidentally matching
    toehold-length set read back a cached result that was actually measured on some
    OTHER file entirely. The hash suffix is what actually guarantees one label maps to
    one exact file: two files with identical bytes get the same label (correctly --
    they *are* the same data), and any byte difference, including something that never
    touches domain lengths at all, changes it."""
    lens = set()
    content = Path(path).read_bytes()
    with Path(path).open(newline="") as handle:
        for raw in csv.DictReader(handle):
            for domain in json.loads(raw["switch_domain_breakdown_json"]):
                if domain["name"] == "toehold":
                    lens.add(domain["end"] - domain["start"])
    digest = hashlib.sha256(content).hexdigest()[:8]
    return f"{gene_name} ({'/'.join(str(n) for n in sorted(lens))}nt, {digest})"


# (path, human tag) pairs -- the tag is for YOUR eyes only (a friendly "this is the
# 4th sele export I've pointed this notebook at" counter, bumped by hand same as
# CSV_SELE_V1's "_v1" suffix above); it is NOT what makes cache entries safe to tell
# apart -- the content hash inside _toehold_length_label does that regardless of
# whether anyone remembers to bump the tag.
CSV_SOURCES = (
    (CSV_SELE_CURRENT, "sele v4"),
)
THREE_CSVS = tuple(_path for _path, _tag in CSV_SOURCES)
THREE_CSV_LABELS = tuple(_toehold_length_label(_path, _tag) for _path, _tag in CSV_SOURCES)
print("THREE_CSV_LABELS:", THREE_CSV_LABELS)


def _nupack_exact_complex_conc(tube_result, strand_names):
    """Total concentration of the complex made of EXACTLY these strands (by name, each
    once), summed across every cyclic strand ordering NUPACK enumerates as a distinct
    complex -- e.g. '(switch+trigger_A+trigger_B)' and '(switch+trigger_B+trigger_A)'
    are two different topological orderings of the SAME three strands, verified
    directly against the real API (both appear, with nonzero and generally different
    concentration, for a plain 3-strand tube), not two complexes to choose between by
    picking whichever a substring check happens to hit first. Exact multiset match on
    the parsed name, never a substring check -- see this notebook's own standing note
    on why 'switch' in name / 'trigger_A' in name can silently match a dimer like
    '(switch+switch+trigger_A)' instead of the intended complex.
    """
    target = sorted(strand_names)
    total = 0.0
    for complex_, conc in tube_result.complex_concentrations.items():
        if sorted(complex_.name.strip("()").split("+")) == target:
            total += conc
    return total


SINGLE_CMP_TOP_N = 150         # NucSyn's own top 150 per CSV (its own rank column, file order) -- the base pool everything below is computed on
SINGLE_CMP_NUPACK_N = 150      # of those, how many get the NUPACK tube+MFE+real-sampling cross-check -- ALL of SINGLE_CMP_TOP_N (benchmarked at ~1.0-1.4s/candidate for the full 10-fake check, so ~150-210s/CSV, ~7-10 min total -- affordable)
SINGLE_CMP_N_SAMPLES = 1200
SINGLE_CMP_THRESHOLD = 0.80
SINGLE_CMP_MAX_WORKERS = 6
SINGLE_CMP_NUPACK_CONC = 1e-7

# CORRECTION (caught after an earlier pass of this cell reported only tube+MFE numbers
# as if they were the full picture): tube analysis + the single MFE structure both
# looked fine for the AND case too, right up until real Boltzmann-sampled ensemble
# structures showed the true picture (this notebook's own AND NUPACK cell above). A
# "% correct" summary built from tube+MFE alone can hide the same failure mode here --
# the true trigger could win the single best (MFE) structure while losing badly across
# the broader ensemble. So this cell now also draws real samples from the folding
# ensemble (NUPACK's own Boltzmann sampler, same diagonal-is-unpaired fix as the
# detailed single-candidate cell above) for every NUPACK-checked candidate, not just
# tube+MFE -- benchmarked first (printed below) to confirm the cost is affordable for
# the full top-SINGLE_CMP_NUPACK_N subset rather than assumed.
SINGLE_CMP_NUPACK_SAMPLE_N = 500   # samples per candidate -- same default the detailed single-candidate cell uses

single_cmp_summary = {}
single_cmp_details = {}

# ---- one-off benchmark: real cost of Boltzmann-sampling one 3-strand single-switch
# complex, on a realistic-size candidate, before committing to sampling the full subset.
_bench_designs, _ = load_ranked_designs(THREE_CSVS[0], folder, limit=2)
if len(_bench_designs) >= 2:
    _bench_model = nupack.Model(material="rna", celsius=TEMPERATURE_C)
    _bs1 = nupack.Strand(_bench_designs[0]["switch"], name="switch")
    _bs2 = nupack.Strand(_bench_designs[0]["trigger"], name="trigger_true")
    _bs3 = nupack.Strand(_bench_designs[1]["trigger"], name="trigger_wrong")
    _bench_complex = nupack.Complex([_bs1, _bs2, _bs3], name="single_cmp_bench")
    _t0 = _time3.time()
    nupack.complex_analysis(complexes=[_bench_complex], model=_bench_model, compute=["sample"],
                             options={"num_sample": SINGLE_CMP_NUPACK_SAMPLE_N})
    _bench_cost = _time3.time() - _t0
    print(f"NUPACK Boltzmann-sampling benchmark: {SINGLE_CMP_NUPACK_SAMPLE_N} samples on one real "
          f"3-strand single-switch complex took {_bench_cost:.2f}s -- sampling all "
          f"{SINGLE_CMP_NUPACK_N} candidates x 3 CSVs would cost ~{_bench_cost * SINGLE_CMP_NUPACK_N * 3:.0f}s, "
          f"affordable, so every NUPACK-checked candidate below gets real ensemble sampling, "
          f"not just tube+MFE.")
else:
    print("benchmark skipped (could not load 2 designs); proceeding with sampling anyway.")


# ---- disk cache: skip the ~9-minute 150-candidate x 3-CSV fake-informed check when
# only a downstream cell (report renderer, scoring formula) changed -- same rationale
# and pattern as the AND mega sweep's own cache below. Versioned filename (v3, was v2 -- bumped when the source CSV changed, so the new file never reads stale cached numbers back under the same rank/label key)
# because a real change to what THIS cell computes (a new metric, a different N_FAKES)
# must bump it, not silently reuse stale data missing the new field -- v2 adds
# sample_coverage/effective_structures per candidate.
# Disk caching removed (was v2/v3/v4 .single_switch_cache_*.json): it silently hid a
# CSV source change behind a fast, wrong cache "hit" once, and was confusing again
# even after the content-hash label fix -- "Run All" must actually run everything,
# every time, with no cached shortcut to reason about. This cell is the ~7-10 min/CSV
# cost that buys you; that cost is now unconditional.

for _label, _csv_path in zip(THREE_CSV_LABELS, THREE_CSVS, strict=True):
    print(f"\n===== SINGLE SWITCH -- {_label}: {_csv_path} =====")
    _designs, _problems = load_ranked_designs(_csv_path, folder, limit=SINGLE_CMP_TOP_N)  # NucSyn's own top-150 pool, unfiltered by any dG re-rank before this point
    for _line_no, _reason in _problems[:5]:
        print(f"  skipped line {_line_no}: {_reason}")

    _rows = []
    for _design in _designs:
        _missing = [d for d in ("toehold", "stem_up", "kozak") if d not in _design["domains"]]
        if _missing:
            continue
        _rows.append({**measure_single_switch(_design, folder), "design": _design})
    _rows.sort(key=lambda r: r["d_ee"])
    _top = _rows[:SINGLE_CMP_TOP_N]

    # ---- ViennaRNA: real P(bound), same SingleStateJob/run_parallel_single as above ----
    _jobs = [
        SingleStateJob(
            key=str(r["design"]["rank"]), switch=r["design"]["switch"], trigger=r["design"]["trigger"],
            footprint=(r["design"]["domains"]["toehold"][0], r["design"]["domains"]["stem_up"][1]),
            temperature=TEMPERATURE_C, n_samples=SINGLE_CMP_N_SAMPLES, threshold=SINGLE_CMP_THRESHOLD,
        )
        for r in _top
    ]
    _started = _time3.time()
    _by_key = run_parallel_single(_jobs, max_workers=SINGLE_CMP_MAX_WORKERS)
    for r in _top:
        _res = _by_key[str(r["design"]["rank"])]
        r["real_bound_fraction"] = _res["bound"]
        # How much of the real Boltzmann ensemble these SINGLE_CMP_N_SAMPLES draws
        # actually represent -- effective-structure-count coverage, not a raw count
        # against the astronomical total structure space. See _sample_coverage's own
        # docstring in _joint_state_parallel.py.
        r["sample_coverage"] = _res["sample_coverage"]
        r["effective_structures"] = _res["effective_structures"]
    _elapsed = _time3.time() - _started
    print(f"  ViennaRNA real P(bound), top {len(_top)}: {_elapsed:.0f}s, "
          f"mean {sum(r['real_bound_fraction'] for r in _top) / len(_top):.1%}, "
          f"mean sample coverage {sum(r['sample_coverage'] for r in _top) / len(_top):.1%}")

    # ---- ViennaRNA: TOEHOLD ALONE, real Boltzmann sampling, no trigger/fake present ----
    # trigger="" folds just the switch (SingleStateJob/_single_worker, updated to omit
    # the "&" entirely rather than being simulated by a NUPACK single-strand tube --
    # that tube metric is near-0 by construction (max_size=1 leaves nowhere else for
    # the concentration to go) and was never a real measurement of whether stem_up
    # actually closes against stem_down with nothing else around. With no second
    # strand, the footprint's only possible partner is its own stem_down, so a HIGH
    # fraction here means the intended OFF-state hairpin is what the ensemble sits
    # in -- a LOW fraction is expected and correct too, since the footprint spans
    # toehold+stem_up together and the toehold portion is supposed to stay
    # single-stranded, capping how high this number can even get for a well-designed
    # switch (same >=threshold-of-the-WHOLE-footprint criterion used everywhere else
    # in this notebook, not a new one).
    _alone_jobs = [
        SingleStateJob(
            key=str(r["design"]["rank"]), switch=r["design"]["switch"], trigger="",
            footprint=(r["design"]["domains"]["toehold"][0], r["design"]["domains"]["stem_up"][1]),
            temperature=TEMPERATURE_C, n_samples=SINGLE_CMP_N_SAMPLES, threshold=SINGLE_CMP_THRESHOLD,
        )
        for r in _top
    ]
    _started = _time3.time()
    _alone_by_key = run_parallel_single(_alone_jobs, max_workers=SINGLE_CMP_MAX_WORKERS)
    for r in _top:
        r["alone_real_paired_fraction"] = _alone_by_key[str(r["design"]["rank"])]["bound"]
    _elapsed = _time3.time() - _started
    print(f"  ViennaRNA real P(footprint paired), toehold ALONE (no trigger/fake), top {len(_top)}: "
          f"{_elapsed:.0f}s, mean {sum(r['alone_real_paired_fraction'] for r in _top) / len(_top):.1%}")

    # ---- NUPACK: tube analysis (alone / +true / +true+wrong) + MFE intended-pair fraction ----
    _nupack_model = nupack.Model(material="rna", celsius=TEMPERATURE_C)
    _subset = _top[:SINGLE_CMP_NUPACK_N]
    _nupack_rows = []
    _started = _time3.time()
    SINGLE_CMP_N_FAKES = 10  # distinct non-overlapping fakes checked per candidate, averaged

    _subset_designs = [x["design"] for x in _subset]
    for _i, r in enumerate(_subset):
        _switch = r["design"]["switch"]
        _true_trig = r["design"]["trigger"]
        _footprint = (r["design"]["domains"]["toehold"][0], r["design"]["domains"]["stem_up"][1])

        _s_switch = nupack.Strand(_switch, name="switch")
        _s_true = nupack.Strand(_true_trig, name="trigger_true")

        # Metrics that do NOT depend on which "wrong" trigger is used -- computed once per
        # candidate, not once per fake.
        _tube_alone = nupack.Tube(strands={_s_switch: SINGLE_CMP_NUPACK_CONC},
                                   complexes=nupack.SetSpec(max_size=1), name="alone")
        _tube_true = nupack.Tube(strands={_s_switch: SINGLE_CMP_NUPACK_CONC, _s_true: SINGLE_CMP_NUPACK_CONC},
                                  complexes=nupack.SetSpec(max_size=2), name="plus_true")
        _tube_base = nupack.tube_analysis(tubes=[_tube_alone, _tube_true], model=_nupack_model)
        _alone_free = _nupack_exact_complex_conc(_tube_base["alone"], ["switch"])
        _leakage_frac = 1.0 - _alone_free / SINGLE_CMP_NUPACK_CONC
        _on_correct = _nupack_exact_complex_conc(_tube_base["plus_true"], ["switch", "trigger_true"])
        _on_correct_frac = _on_correct / SINGLE_CMP_NUPACK_CONC

        # SINGLE_CMP_N_FAKES distinct, non-overlapping "wrong" designs -- deterministic
        # rotation (never randomness, CLAUDE.md sec 6), each already-picked rank excluded so
        # this candidate's own set of 10 never repeats one fake.
        _picked_ranks = {r["design"]["rank"]}
        _wrong_designs = []
        for _k in range(SINGLE_CMP_N_FAKES):
            _wd = pick_fake_design(
                r["design"], r["design"], set(_picked_ranks), _subset_designs,
                start_offset=_i + 1 + _k,
            )
            if _wd is None:
                break  # pool exhausted -- fewer than SINGLE_CMP_N_FAKES non-overlapping designs exist
            _picked_ranks.add(_wd["rank"])
            _wrong_designs.append(_wd)

        _per_fake_cross = []
        _per_fake_mfe_intended = []
        _per_fake_ensemble = []
        _per_fake_wrong_only_nupack = []
        _per_fake_wrong_only_vienna = []

        # ---- ViennaRNA fake-only checks: ONE pool for all of this candidate's fakes ----
        # Was one run_parallel_single([single_job], max_workers=1) call per fake, nested
        # two loops deep: 150 candidates x 10 fakes = 1500 process-pool spin-ups, each
        # forking a worker and re-importing RNA to run exactly one fold, with
        # max_workers=1 so it got zero parallelism for all that overhead. Measured at
        # 40 ms/job that way vs 7 ms/job batched (5.8x). Batched per candidate here:
        # 10 jobs per pool call, 150 calls instead of 1500.
        _fake_only_jobs = [
            SingleStateJob(
                key=f"fakeonly{_fk}", switch=_switch, trigger=_wd["trigger"],
                footprint=_footprint, temperature=TEMPERATURE_C,
                n_samples=SINGLE_CMP_N_SAMPLES, threshold=SINGLE_CMP_THRESHOLD,
            )
            for _fk, _wd in enumerate(_wrong_designs)
        ]
        _fake_only_by_key = run_parallel_single(_fake_only_jobs, max_workers=SINGLE_CMP_MAX_WORKERS)

        for _fk, _wd in enumerate(_wrong_designs):
            _wrong_trig = _wd["trigger"]
            _s_wrong = nupack.Strand(_wrong_trig, name="trigger_wrong")

            _tube_all = nupack.Tube(strands={_s_switch: SINGLE_CMP_NUPACK_CONC, _s_true: SINGLE_CMP_NUPACK_CONC,
                                              _s_wrong: SINGLE_CMP_NUPACK_CONC},
                                     complexes=nupack.SetSpec(max_size=3), name="all")
            # FAKE-ONLY tube: true trigger ABSENT -- does an off-target trigger alone
            # spuriously open this switch?
            _tube_wrong_only = nupack.Tube(strands={_s_switch: SINGLE_CMP_NUPACK_CONC,
                                                      _s_wrong: SINGLE_CMP_NUPACK_CONC},
                                            complexes=nupack.SetSpec(max_size=2), name="plus_wrong_only")
            _tube_result = nupack.tube_analysis(tubes=[_tube_all, _tube_wrong_only], model=_nupack_model)

            _cross_correct = _nupack_exact_complex_conc(_tube_result["all"], ["switch", "trigger_true"])
            _per_fake_cross.append(_cross_correct / SINGLE_CMP_NUPACK_CONC)

            _wrong_only_bound = _nupack_exact_complex_conc(_tube_result["plus_wrong_only"], ["switch", "trigger_wrong"])
            _per_fake_wrong_only_nupack.append(_wrong_only_bound / SINGLE_CMP_NUPACK_CONC)

            # Same fake-only question, real ViennaRNA Boltzmann sampling -- already
            # computed for every fake in the single batched pool call above.
            _per_fake_wrong_only_vienna.append(_fake_only_by_key[f"fakeonly{_fk}"]["bound"])

            # MFE intended-pair fraction, 3-strand (switch+true+wrong) complex, THIS fake.
            _complex = nupack.Complex([_s_switch, _s_true, _s_wrong],
                                       name=f"single_cmp_{_label}_{_i}_{_fk}")
            _mfe = nupack.complex_analysis(complexes=[_complex], model=_nupack_model, compute=["mfe"])[_complex].mfe[0]
            _n_off = len(_switch)
            _true_span = (_n_off, _n_off + len(_true_trig))
            _pairs = [(int(a), int(b)) for a, b in zip(*_mfe.structure.matrix().nonzero(), strict=True) if a < b]
            _intended = sum(
                1 for a, b in _pairs
                if (_footprint[0] <= a < _footprint[1] and _true_span[0] <= b < _true_span[1])
                or (_footprint[0] <= b < _footprint[1] and _true_span[0] <= a < _true_span[1])
            )
            _footprint_len = _footprint[1] - _footprint[0]
            _per_fake_mfe_intended.append(_intended / _footprint_len)

            # Real Boltzmann-sampled structures for THIS fake (not just tube+MFE) -- same
            # diagonal-is-unpaired fix as every other NUPACK sampling check in this notebook.
            _sample_result = nupack.complex_analysis(
                complexes=[_complex], model=_nupack_model, compute=["sample"],
                options={"num_sample": SINGLE_CMP_NUPACK_SAMPLE_N},
            )[_complex]

            def _single_cmp_frac_paired(sample_structure, span):
                diag = sample_structure.matrix().diagonal()
                return float((diag[span[0]:span[1]] == 0).sum()) / (span[1] - span[0])

            _n_sampled = len(_sample_result.sample)
            _ensemble_bound = sum(
                1 for _s in _sample_result.sample
                if _single_cmp_frac_paired(_s, _footprint) >= SINGLE_CMP_THRESHOLD
            )
            _per_fake_ensemble.append(_ensemble_bound / _n_sampled)

        def _mean_or_none(xs):
            return sum(xs) / len(xs) if xs else None

        _nupack_rows.append({
            "rank": r["design"]["rank"], "leakage_frac": _leakage_frac,
            "on_correct_frac": _on_correct_frac,
            "n_fakes_checked": len(_wrong_designs),
            # Every one of these is now a MEAN over n_fakes_checked distinct, non-
            # overlapping fake triggers, not one draw -- the corresponding "_per_fake" list
            # is kept alongside for transparency (individual draws, not just the average).
            "cross_correct_frac_tube_mfe_only": _mean_or_none(_per_fake_cross),
            "cross_correct_frac_tube_mfe_only_per_fake": _per_fake_cross,
            "mfe_intended_frac_tube_mfe_only": _mean_or_none(_per_fake_mfe_intended),
            "mfe_intended_frac_tube_mfe_only_per_fake": _per_fake_mfe_intended,
            "ensemble_bound_frac": _mean_or_none(_per_fake_ensemble),
            "ensemble_bound_frac_per_fake": _per_fake_ensemble,
            "ensemble_n_samples": SINGLE_CMP_NUPACK_SAMPLE_N,
            "fake_only_nupack_bound_frac": _mean_or_none(_per_fake_wrong_only_nupack),
            "fake_only_nupack_bound_frac_per_fake": _per_fake_wrong_only_nupack,
            "fake_only_vienna_bound_frac": _mean_or_none(_per_fake_wrong_only_vienna),
            "fake_only_vienna_bound_frac_per_fake": _per_fake_wrong_only_vienna,
        })
    _elapsed = _time3.time() - _started
    print(f"  NUPACK top {len(_nupack_rows)} (tube + MFE + {SINGLE_CMP_NUPACK_SAMPLE_N}-sample Boltzmann "
          f"ensemble each): {_elapsed:.0f}s ({_elapsed / len(_nupack_rows):.1f}s/candidate)")


    _mean_leak = sum(x["leakage_frac"] for x in _nupack_rows) / len(_nupack_rows)
    _mean_on = sum(x["on_correct_frac"] for x in _nupack_rows) / len(_nupack_rows)
    _mean_cross_tube_mfe = sum(x["cross_correct_frac_tube_mfe_only"] for x in _nupack_rows) / len(_nupack_rows)
    _mean_mfe_tube_mfe = sum(x["mfe_intended_frac_tube_mfe_only"] for x in _nupack_rows) / len(_nupack_rows)
    _mean_ensemble = sum(x["ensemble_bound_frac"] for x in _nupack_rows) / len(_nupack_rows)
    _n_ensemble_ge50 = sum(1 for x in _nupack_rows if x["ensemble_bound_frac"] >= 0.5)
    print(f"    OFF-alone leakage {_mean_leak:.1%}  |  ON tube correct {_mean_on:.1%}")
    print(f"    [tube+MFE only, NOT ensemble-validated] true+wrong tube correct {_mean_cross_tube_mfe:.1%}  |  "
          f"MFE intended-pair frac {_mean_mfe_tube_mfe:.1%}")
    print(f"    [REAL Boltzmann-sampled ensemble, {SINGLE_CMP_NUPACK_SAMPLE_N} samples/candidate] "
          f"mean P(true footprint bound, wrong trigger present) = {_mean_ensemble:.1%}  "
          f"({_n_ensemble_ge50}/{len(_nupack_rows)} candidates >= 50%)")

    single_cmp_summary[_label] = {
        "csv": _csv_path, "n_top": len(_top),
        "vienna_mean_bound": sum(r["real_bound_fraction"] for r in _top) / len(_top),
        "vienna_n_ge50": sum(1 for r in _top if r["real_bound_fraction"] >= 0.5),
        "nupack_n": len(_nupack_rows), "nupack_mean_leakage": _mean_leak,
        "nupack_mean_on_correct": _mean_on,
        "nupack_mean_cross_correct_tube_mfe_only": _mean_cross_tube_mfe,
        "nupack_mean_mfe_intended_tube_mfe_only": _mean_mfe_tube_mfe,
        "nupack_mean_ensemble_bound": _mean_ensemble,
        "nupack_n_ensemble_ge50": _n_ensemble_ge50,
    }
    single_cmp_details[_label] = {"top": _top, "nupack_rows": _nupack_rows}

print(f"\n===== SINGLE SWITCH: three-CSV comparison (top {SINGLE_CMP_TOP_N} each; NUPACK -- tube+MFE+real sampling -- on all {SINGLE_CMP_NUPACK_N}) =====")
print("CLEAN baseline (switch + its own real trigger ONLY, no fake trigger present):")
print("  'ViennaRNA P(bound)' and 'NUPACK [CLEAN] ON tube correct' below.")
print("WITH-FAKE-TRIGGER leakage numbers (same switch, its true trigger, AND one other real, wrong")
print("trigger from the same pool all present together) -- a direct before/after pairing against the")
print("clean columns above, not a standalone number:")
print("  '[+fake, tube+MFE only]' columns are the single lowest-energy structure + tube concentration --")
print("  the SAME kind of check that looked fine for the AND case before real ensemble sampling showed")
print("  the true picture there. '[+fake, ENSEMBLE] P(bound)' is the real, ensemble-validated signal:")
print(f"  {SINGLE_CMP_NUPACK_SAMPLE_N} independently Boltzmann-sampled structures per candidate, true trigger")
print("  + one wrong trigger both present, same >=80%-of-footprint-paired criterion used throughout this notebook.")
print(f"{'csv':<20}{'[CLEAN] ViennaRNA':>18}{'[CLEAN] NUPACK ON':>18}{f'>=50% of {SINGLE_CMP_TOP_N}':>13}"
      f"{'[+fake tube+MFE]ok':>19}{'[+fake tube+MFE]MFE':>20}{'[+fake ENSEMBLE]':>17}")
for _label in THREE_CSV_LABELS:
    s = single_cmp_summary[_label]
    print(f"{_label:<20}{s['vienna_mean_bound']:>17.1%} {s['nupack_mean_on_correct']:>17.1%} "
          f"{s['vienna_n_ge50']:>12d} "
          f"{s['nupack_mean_cross_correct_tube_mfe_only']:>18.1%} "
          f"{s['nupack_mean_mfe_intended_tube_mfe_only']:>19.1%} "
          f"{s['nupack_mean_ensemble_bound']:>16.1%}")


THREE_CSV_LABELS: ('sele v4 (15/18/21nt, 804b5213)',)
NUPACK Boltzmann-sampling benchmark: 500 samples on one real 3-strand single-switch complex took 0.03s -- sampling all 150 candidates x 3 CSVs would cost ~12s, affordable, so every NUPACK-checked candidate below gets real ensemble sampling, not just tube+MFE.

===== SINGLE SWITCH -- sele v4 (15/18/21nt, 804b5213): /Users/zeev/Projects/TOEHOLDS/sele/basic_switch/results_ranked.csv =====


[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2


  ViennaRNA real P(bound), top 150: 1s, mean 95.0%, mean sample coverage 100.0%


[ERROR]   backtracking failed for qm2_real 2


  ViennaRNA real P(footprint paired), toehold ALONE (no trigger/fake), top 150: 1s, mean 2.7%


[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtr

  NUPACK top 150 (tube + MFE + 500-sample Boltzmann ensemble each): 206s (1.4s/candidate)
    OFF-alone leakage -0.0%  |  ON tube correct 99.5%
    [tube+MFE only, NOT ensemble-validated] true+wrong tube correct 96.9%  |  MFE intended-pair frac 95.8%
    [REAL Boltzmann-sampled ensemble, 500 samples/candidate] mean P(true footprint bound, wrong trigger present) = 99.9%  (150/150 candidates >= 50%)

===== SINGLE SWITCH: three-CSV comparison (top 150 each; NUPACK -- tube+MFE+real sampling -- on all 150) =====
CLEAN baseline (switch + its own real trigger ONLY, no fake trigger present):
  'ViennaRNA P(bound)' and 'NUPACK [CLEAN] ON tube correct' below.
WITH-FAKE-TRIGGER leakage numbers (same switch, its true trigger, AND one other real, wrong
trigger from the same pool all present together) -- a direct before/after pairing against the
clean columns above, not a standalone number:
  '[+fake, tube+MFE only]' columns are the single lowest-energy structure + tube concentration --
  the SAME k

In [106]:
# ==================================================================================
# SINGLE-SWITCH SCORING -- one fake-informed score, computed once per CSV over its own
# full SINGLE_CMP_TOP_N (150) pool, reused by everything downstream: the single-switch
# report top-20 AND the per-CSV top-20 design pool the AND side builds its 20x20 grid
# from. One function, not two competing scores for "the same" ranking.
#
# ON_effective  = min(clean P(bound), P(bound) with one wrong trigger ALSO present) --
#   the worse of "does it turn on at all" and "does it stay on with noise nearby".
# OFF_effective = max(REAL toehold-alone leak, clean-tube alone leak, fake-ONLY leak
#   -- wrong trigger present, true trigger absent) -- the worst of three ways this
#   candidate could fail to stay off: spontaneously (nothing present at all, real
#   Boltzmann-sampled), the NUPACK tube's own alone-state number, and with only an
#   off-target trigger nearby. Same worst-of-many principle this notebook already
#   uses for the AND side's leak metric (a design cannot hide a bad number behind a
#   good one). "REAL toehold-alone leak" = 1 - alone_real_paired_fraction (the cell
#   above's real pbacktrack sampling of the switch by itself, no second strand) --
#   the number the NUPACK tube metric was never actually measuring (see that cell's
#   own docstring).
# SCORE = ON_effective - OFF_effective.
#
# Every one of these five numbers is itself already a MEAN over
# SINGLE_CMP_N_FAKES=10 distinct, non-overlapping fake triggers (computed by the cell
# above) -- this is not adding a new measurement, only a new combination of ones
# already taken.
# ==================================================================================
def _single_fake_informed_score(row, fc):
    """Returns (on_eff, off_eff, score), or None if any required fake-check number was
    never measured (CLAUDE.md sec 3: no default standing in for a measurement not
    taken -- a candidate this happens to is simply left out of the ranking, not scored
    as if it were clean)."""
    on_clean = row["real_bound_fraction"]
    on_fake = fc["ensemble_bound_frac"]
    off_alone_real = row.get("alone_real_paired_fraction")
    off_clean = fc["leakage_frac"]
    off_fake = fc["fake_only_vienna_bound_frac"]
    if None in (on_clean, on_fake, off_alone_real, off_clean, off_fake):
        return None
    on_eff = min(on_clean, on_fake)
    off_eff = max(1.0 - off_alone_real, off_clean, off_fake)
    return on_eff, off_eff, on_eff - off_eff


single_scored_by_csv = {}
for _csv_label in THREE_CSV_LABELS:
    _nupack_by_rank = {row["rank"]: row for row in single_cmp_details[_csv_label]["nupack_rows"]}
    _scored = []
    for _row in single_cmp_details[_csv_label]["top"]:
        _fc = _nupack_by_rank.get(_row["design"]["rank"])
        if _fc is None:
            continue
        _result = _single_fake_informed_score(_row, _fc)
        if _result is None:
            continue
        _on_eff, _off_eff, _score = _result
        _scored.append({
            "csv_label": _csv_label, "row": _row, "fake_check": _fc,
            "on_eff": _on_eff, "off_eff": _off_eff, "score": _score,
        })
    _scored.sort(key=lambda e: -e["score"])
    single_scored_by_csv[_csv_label] = _scored

single_top20_by_csv = {
    _csv_label: [e["row"]["design"] for e in _scored[:20]]
    for _csv_label, _scored in single_scored_by_csv.items()
}

for _csv_label in THREE_CSV_LABELS:
    _scored = single_scored_by_csv[_csv_label]
    print(f"\n{_csv_label}: {len(_scored)}/{SINGLE_CMP_TOP_N} candidates fully scored "
          f"(fake-informed ON/OFF) -- top 20 of those become this CSV's AND pool")
    print(f"{'#':>3}  {'rank':>5}  {'ON eff':>7}  {'OFF eff':>8}  {'score':>7}")
    for _i, _e in enumerate(_scored[:20], start=1):
        print(f"{_i:>3}  {_e['row']['design']['rank']:>5}  {_e['on_eff']:>7.1%}  "
              f"{_e['off_eff']:>8.1%}  {_e['score']:>+7.1%}")


sele v4 (15/18/21nt, 804b5213): 150/150 candidates fully scored (fake-informed ON/OFF) -- top 20 of those become this CSV's AND pool
  #   rank   ON eff   OFF eff    score
  1    143    97.5%     74.8%   +22.7%
  2     84   100.0%     78.2%   +21.7%
  3     37   100.0%     82.2%   +17.8%
  4     89   100.0%     84.8%   +15.1%
  5     30   100.0%     86.6%   +13.4%
  6    130    99.9%     87.4%   +12.5%
  7    103   100.0%     88.2%   +11.7%
  8     65    99.9%     88.4%   +11.5%
  9     68    98.9%     89.2%    +9.7%
 10     44    99.8%     90.6%    +9.2%
 11    150    99.2%     90.1%    +9.1%
 12      5   100.0%     91.8%    +8.2%
 13     49   100.0%     92.4%    +7.6%
 14     87    86.3%     79.2%    +7.1%
 15     86   100.0%     93.0%    +7.0%
 16     13    99.1%     92.7%    +6.4%
 17     85   100.0%     93.8%    +6.1%
 18    123    98.4%     92.7%    +5.8%
 19     42   100.0%     94.4%    +5.6%
 20     93    99.7%     94.2%    +5.4%


# Part 2 — the AND switch

Two of Part 1's single switches fused into one two-input construct: the 5' half keeps
everything up to and including its own hairpin, the 3' half keeps everything from its own
hairpin onward, so one prefix, both hairpins in series, one Kozak. Everything below —
`fuse()`, `measure()`, the run cell and the AND report — belongs to this part. Output:
`var/reports/and_toehold_report_<geneA>_<geneB>.html/.pdf`.

In [107]:
# ===== PART 2 -- AND switch =====
def fuse(design_5p, design_3p, cut_before=CUT_5P_BEFORE, keep_after=KEEP_3P_AFTER, spacer=SPACER_SEQ):
    """One AND candidate: every domain from both halves, relocated into the fused frame.

    Returns ``(built, problem)``. ``built["domains_5p"]``/``["domains_3p"]`` carry every
    surviving domain's fused-frame span, not just the toehold — the layout bar and the
    register diagrams both need to know where every domain landed, and re-deriving those
    offsets in two more places would be two more chances to get the arithmetic wrong.
    ``problem`` is non-None when either design lacks the domain its cut is defined by,
    lacks ``stem_up`` (the ascending arm a trigger's footprint needs), or when a relocated
    span does not hold the sequence it started as — slicing a guessed offset instead would
    still return a sequence, and it would fold into a plausible-looking wrong answer
    nothing downstream could catch.

    Note for other topologies: cutting the 5' half at ``kozak`` leaves its ``pre_kozak``
    arm in place while dropping the ``post_kozak`` it was designed to pair with. Both are
    zero-length in ``euk_open_5p_kozak_after_stem``, so nothing is orphaned here; a
    topology that uses them needs this cut reconsidered, not just re-pointed.

    ``spacer`` sits between the two bodies unshifted-domain-wise for the 5' half and
    folded into the 3' half's shift -- a physical separator between the two hairpins,
    not part of either design's own domain set (SPACER_SEQ's own docstring covers why
    this particular sequence was chosen).
    """
    for label, design, domain in (("5'", design_5p, cut_before), ("3'", design_3p, keep_after)):
        if domain not in design["domains"]:
            return None, f"{label} line {design['line_no']}: no {domain!r} domain"

    cut_5p = design_5p["domains"][cut_before][0]
    keep_3p = design_3p["domains"][keep_after][1]
    body_5p = design_5p["switch"][:cut_5p]
    body_3p = design_3p["switch"][keep_3p:]
    fused = body_5p + spacer + body_3p
    shift = len(body_5p) + len(spacer) - keep_3p

    # Every 5' domain that ends at or before the cut survives unshifted; every 3' domain
    # that starts at or after the keep point survives, shifted by how far it moved.
    domains_5p = {name: span for name, span in design_5p["domains"].items() if span[1] <= cut_5p}
    domains_3p = {
        name: (start + shift, end + shift)
        for name, (start, end) in design_3p["domains"].items()
        if start >= keep_3p
    }

    # CLAUDE.md §6: every coordinate translation is an off-by-one until it is checked.
    for label, domains, design, base_shift in (
        ("5'", domains_5p, design_5p, 0),
        ("3'", domains_3p, design_3p, shift),
    ):
        for name, (start, end) in domains.items():
            original_start, original_end = start - base_shift, end - base_shift
            if fused[start:end] != design["switch"][original_start:original_end]:
                return None, f"{label} line {design['line_no']}: {name} does not survive relocation"

    for label, domains, design in (("5'", domains_5p, design_5p), ("3'", domains_3p, design_3p)):
        missing = [name for name in ("toehold", "stem_up") if name not in domains]
        if missing:
            return None, f"{label} line {design['line_no']}: missing {', '.join(missing)} after the cut"

    return (
        {
            "fused": fused,
            "domains_5p": domains_5p,
            "domains_3p": domains_3p,
            "toehold_5p": domains_5p["toehold"],
            "toehold_3p": domains_3p["toehold"],
            # The footprint is toehold + the ascending arm -- exactly the region the
            # trigger reverse-complements against, same definition the single-input
            # report uses for its own "footprint" highlight.
            "footprint_5p": (domains_5p["toehold"][0], domains_5p["stem_up"][1]),
            "footprint_3p": (domains_3p["toehold"][0], domains_3p["stem_up"][1]),
            "spacer_span": (len(body_5p), len(body_5p) + len(spacer)),
        },
        None,
    )

In [108]:
# ===== PART 2 -- AND switch =====
# _mean_unpaired is already imported in the loader cell above (module-private in
# engine.gates.toehold; a notebook reaching for it directly is the honest way to stay
# consistent with the engine rather than re-deriving the same accessibility arithmetic --
# CLAUDE.md §1). motif_self_bind_score is defined there too.


def measure(built, trigger_5p, trigger_3p, folder, own_kozak_rc_3p, own_prefix_rc_5p):
    """Energies for the four states, both toehold accessibilities, and the Kozak/prefix
    self-binding risk of whichever toehold each fixed element could actually reach.

    ``FoldEngine`` caches per sequence, so the shared instance built in the run cell below
    is the only one that should ever fold here — a second instance means a cold cache and,
    worse, a second chance to fold at a different temperature.

    ``own_kozak_rc_3p``/``own_prefix_rc_5p`` are the 3'/5' design's own
    ``metrics["own_kozak_rc_score"]``/``["own_prefix_rc_score"]`` (``load_ranked_designs``),
    passed in rather than recomputed: fusion never changes the 3' half's own toehold+Kozak,
    or the 5' half's own toehold+prefix -- only their position in the fused frame -- and
    ``motif_self_bind_score`` depends only on sequence content, so recomputing either here
    was verified to return the exact same number the load-time pass already computed. Only
    the *other* toehold's score against each surviving element is genuinely
    fusion-dependent (5' toehold vs the surviving Kozak; 3' toehold vs the surviving
    prefix), since each depends on which design it ended up paired with — those two are
    still computed here, per pair.
    """
    fused = built["fused"]
    states = {
        "off": fused,  # no trigger: both hairpins closed
        "t5": f"{fused}&{trigger_5p}",  # '&' = a true multi-strand complex, not a fusion
        "t3": f"{fused}&{trigger_3p}",
        "both": f"{fused}&{trigger_5p}&{trigger_3p}",
    }

    out = {}
    for name, strands in states.items():
        out[f"mfe_{name}"] = folder.mfe(strands).energy
        out[f"ee_{name}"] = folder.partition(strands)

    # dG of a state = that state minus the trigger-free OFF state, so a negative number is
    # the stabilisation that adding trigger(s) bought.
    for name in ("t5", "t3", "both"):
        out[f"d_mfe_{name}"] = out[f"mfe_{name}"] - out["mfe_off"]
        out[f"d_ee_{name}"] = out[f"ee_{name}"] - out["ee_off"]

    # Reported, never ranked on: how much the second trigger buys beyond whichever single
    # trigger already did most of the work. Near zero means one trigger alone already
    # opens the construct -- an OR wearing an AND's shape, which is the specific failure
    # this architecture has to be checked for (see this notebook's mechanism section).
    out["and_margin_ee"] = min(out["d_ee_t5"], out["d_ee_t3"]) - out["d_ee_both"]

    # The ranking key. d_ee_both alone only asks "is ON more stable than OFF" -- it says
    # nothing about whether the ensemble would rather sit in some *other* not-fully-on
    # state instead, e.g. "+trigger B alone" being more stable than plain OFF. The real
    # competitor to the ON state is whichever of the three not-fully-activated states
    # (OFF, +A alone, +B alone) is thermodynamically most favourable -- the most negative
    # of the three, since a system settles toward lower free energy. worst_alt_ee is that
    # state's energy; d_ee_worst_alt is how far ON beats it. A design can look good on
    # d_ee_both alone yet have a mediocre d_ee_worst_alt, if one single trigger already
    # competes closely with full activation.
    worst_alt_ee = min(out["ee_off"], out["ee_t5"], out["ee_t3"])
    out["d_ee_worst_alt"] = out["ee_both"] - worst_alt_ee

    # Both toeholds have to be open in the OFF state for either trigger to nucleate, so
    # the limiting one is the one that decides whether this construct can start at all.
    off_matrix = folder.base_pair_probabilities(fused)
    out["access_5p"] = _mean_unpaired(off_matrix, *built["toehold_5p"])
    out["access_3p"] = _mean_unpaired(off_matrix, *built["toehold_3p"])
    out["access_min"] = min(out["access_5p"], out["access_3p"])

    # Whichever Kozak actually survives fusion (always the 3' half's -- the 5' half's own
    # copy was cut away in fuse()) is the one the 5' toehold could realistically
    # hybridise to in the finished molecule -- its own Kozak is gone, but nothing stops it
    # from reaching the surviving one instead: intramolecular pairing does not require
    # adjacency. A real ViennaRNA fold earlier this session found exactly this kind of
    # pairing spanning ~50 nt of intervening hairpin, so "less important" for the 5'
    # toehold is not "not checked".
    fused_kozak_span = built["domains_3p"].get("kozak")
    if fused_kozak_span is not None:
        kozak_seq = fused[fused_kozak_span[0] : fused_kozak_span[1]]
        toehold_5p_seq = fused[built["toehold_5p"][0] : built["toehold_5p"][1]]
        out["kozak_rc_5p"] = motif_self_bind_score(toehold_5p_seq, kozak_seq)
    else:
        out["kozak_rc_5p"] = None
    out["kozak_rc_3p"] = own_kozak_rc_3p

    # Mirror check, other direction: the prefix always survives fusion -- it belongs to
    # the 5' half, and fuse() never cuts it (only the 3' half's prefix is cut, in the
    # KEEP_3P_AFTER step). So it's the 3' toehold that could realistically reach back and
    # hybridise to it, same "distance is not protection" reasoning as the Kozak check
    # above, just pointed the other way down the molecule.
    fused_prefix_span = built["domains_5p"].get("prefix")
    if fused_prefix_span is not None:
        prefix_seq = fused[fused_prefix_span[0] : fused_prefix_span[1]]
        toehold_3p_seq = fused[built["toehold_3p"][0] : built["toehold_3p"][1]]
        out["prefix_rc_3p"] = motif_self_bind_score(toehold_3p_seq, prefix_seq)
    else:
        out["prefix_rc_3p"] = None
    out["prefix_rc_5p"] = own_prefix_rc_5p
    return out

## Fake-trigger leakage sweep — all three CSVs x spacer on/off

Generalizes the FULL LEAKAGE/SPECIFICITY SWEEP cell above (`fuse()`, `measure()`,
`FakeSweepJob`/`run_parallel_fake_sweep`, `_pick_fake_seq` — all reused) across all
three source CSVs (`THREE_CSVS`) and both spacer configurations (`spacer=""` and
`spacer=SPACER_SEQ`), each on its own top-20 pool. For every pair the real joint
`+A+B sampled_both` (does it actually work as an AND) is reported next to the `alone`
state's own open-fractions (does either footprint leak open with nothing present) —
never one without the other, per this session's own finding that a high "both open"
number can be produced by a chronically-open footprint rather than genuine gating.


In [109]:
# ==================================================================================
# MEGA FAKE-LEAKAGE SWEEP -- all three source CSVs x both spacer configs, each on its
# own top-20 pool. Same real logic as the FULL LEAKAGE/SPECIFICITY SWEEP cell above
# (fuse(), measure(), FakeSweepJob/run_parallel_fake_sweep, _pick_fake_seq -- all reused
# from earlier cells, nothing re-implemented) -- generalised over (csv, spacer) instead
# of hard-coding CSV_SELE + a single spacer choice. THREE_CSVS/THREE_CSV_LABELS come
# from the PART 1 three-CSV comparison cell above; each CSV's pool is now
# `single_top20_by_csv[csv_label]` (the fake-informed top 20 the scoring cell above
# selected: min(clean ON, ON-with-a-fake) - max(clean OFF-leak, fake-only-leak) among
# NucSyn's own top-150), not a blind `load_ranked_designs(..., limit=20)` of NucSyn's
# raw top 20 -- pools are never shared or cross-paired between CSVs, same rule this
# notebook already applies to sele self-pairing (each CSV is its own self-contained
# sele-vs-sele comparison).
#
# WHY BOTH SPACER CONFIGS: the earlier single-CSV fake-sweep cell ran only with
# spacer=SPACER_SEQ, after a code comment recorded "results without a spacer were not
# promising". This sweep checks that claim is not itself an artifact of one CSV/pool,
# and puts the no-spacer number in the SAME table as the spacer number for a fair
# side-by-side, instead of relying on a comment about a run that is not reproduced here.
#
# HOW TO RE-RUN THIS YOURSELF: run every cell through the PART 1 three-CSV comparison
# cell above first (needs `folder`, `load_ranked_designs`, `fuse`, `measure`,
# `MIN_TRIGGER_SEPARATION_NT`, `TEMPERATURE_C`, `THREE_CSVS`, `THREE_CSV_LABELS`,
# `SPACER_SEQ` -- all already defined above). Adjust the parameters below, then run
# just this cell.
#
# COST: benchmarked earlier this session at ~60-90s per (CSV, spacer) combination with
# MEGA_POOL_SIZE=20, MEGA_N_SAMPLES=1200, MEGA_MAX_WORKERS=6 (~308-354 valid pairs after
# trigger-overlap exclusion) -- 3 CSVs x 2 spacer configs is ~10-14 minutes of real
# compute, including the per-pair dG pass. MEGA_PAIR_LIMIT caps pairs per combination
# for a quick smoke test; None runs every valid pair, which is what the published
# report uses. Every state's full result (not just +A+B) is kept per pair, including
# the fake-only states (+fake1, +fake2, +fake1+fake2, +A+fake1, +B+fake2,
# +A+B+fake1+fake2) -- the report renderer decides what to show, this cell always
# computes and stores the complete FAKE_SWEEP_STATES set.
# ==================================================================================
import json as _json_mega
import sys as _sys_mega51
import time as _time_mega
from pathlib import Path as _Path_mega51

_here_mega51 = _Path_mega51.cwd()
for _base in (_here_mega51, *_here_mega51.parents):
    if (_base / "_joint_state_parallel.py").exists():
        if str(_base) not in _sys_mega51.path:
            _sys_mega51.path.insert(0, str(_base))
        break
from _joint_state_parallel import (
    FakeSweepJob,
    generate_spacer_candidates,
    pick_fake_design,
    pick_optimized_spacer,
    run_parallel_fake_sweep,
)

# ---- parameters -----------------------------------------------------------------
MEGA_POOL_SIZE = 20            # size of single_top20_by_csv[...] -- documented here, not re-derived
MEGA_N_SAMPLES = 1200
MEGA_THRESHOLD = 0.80
MEGA_MAX_WORKERS = 6
MEGA_SPACER_LABELS = ("no_spacer", "spacer")   # "no_spacer" = fuse(spacer=""); "spacer" = optimized per pair below

# Per-pair spacer optimization: SPACER_SEQ (the old fixed global spacer) is now only a
# fallback default for fuse()'s own signature, not what the "spacer" config actually
# uses. Its own docstring's claim of being pre-verified against "a real trigger" turned
# out to only ever have been checked against ONE trigger -- this notebook's own
# mechanism investigation found it DOES have real accidental complementarity to at
# least one real toehold, which is exactly the failure mode a fixed global spacer
# cannot see coming and a per-pair search can screen for directly. 15-20nt, picked per
# pair to minimize the longest complementary run against EITHER pair's toehold or
# trigger (cheap string matching, no folding -- see pick_optimized_spacer's own
# docstring), low-GC tie-broken. The real ON/OFF verification is still the same
# Boltzmann-sampled FakeSweepJob sweep every spacer choice goes through below -- this
# search only picks a better STARTING candidate, it does not replace that check.
SPACER_MIN_LENGTH = 15
SPACER_MAX_LENGTH = 20
SPACER_CANDIDATE_POOL_SIZE = 150   # benchmarked at ~0.012s/pair to screen -- ~13s for the whole 3-CSV sweep, negligible next to the ~10-14 min fold/sample cost
SPACER_SEED = 1                    # literal constant, never wall-clock (CLAUDE.md sec 6) -- same candidate pool every run
SPACER_CANDIDATES = generate_spacer_candidates(
    SPACER_CANDIDATE_POOL_SIZE, SPACER_MIN_LENGTH, SPACER_MAX_LENGTH, seed=SPACER_SEED
)
# None = every valid pair per (csv, spacer) combination; an int caps it for a quick
# smoke test before committing to the full run.
MEGA_PAIR_LIMIT = None

# Disk caching removed (was v1-v4 .mega_fake_sweep_cache_*.json), same reasoning as
# the single-switch cache above: "Run All" must actually run everything, every time.
mega_sweep_results = {}   # (csv_label, spacer_label) -> list of per-pair result dicts
mega_sweep_meta = {}      # (csv_label, spacer_label) -> {"n_pairs",
                           # "n_skipped_no_fake", "elapsed_s"}

_mega_started_all = _time_mega.time()
for _csv_label, _csv_path in zip(THREE_CSV_LABELS, THREE_CSVS, strict=True):
    _mega_pool = single_top20_by_csv[_csv_label]
    assert len(_mega_pool) == MEGA_POOL_SIZE, (
        f"{_csv_label}: expected {MEGA_POOL_SIZE} fake-informed candidates, "
        f"got {len(_mega_pool)} -- SINGLE_CMP_TOP_N pool may be too small or too "
        f"many candidates failed the fake-informed score (see the cell above)"
    )
    print(f"\n===== {_csv_label}: top {len(_mega_pool)} designs (fake-informed score) "
          f"(ranks {[d['rank'] for d in _mega_pool]}) =====")

    for _spacer_label in MEGA_SPACER_LABELS:
        _mega_jobs = []
        _mega_built_by_key = {}
        _mega_skipped_no_fake = 0
        _mega_excluded = 0
        _mega_spacer_worst_runs = []  # for the post-sweep sanity print below
        for _i5, _d5 in enumerate(_mega_pool):
            for _i3, _d3 in enumerate(_mega_pool):
                if _d5["rank"] == _d3["rank"]:
                    continue
                _a0, _a1 = _d5["trigger_start"], _d5["trigger_end"]
                _b0, _b1 = _d3["trigger_start"], _d3["trigger_end"]
                if min(_a1, _b1) - max(_a0, _b0) >= MIN_TRIGGER_SEPARATION_NT:
                    _mega_excluded += 1
                    continue
                if _spacer_label == "spacer":
                    _pair_spacer, _pair_spacer_worst = pick_optimized_spacer(
                        _d5, _d3, SPACER_CANDIDATES
                    )
                    _mega_spacer_worst_runs.append(_pair_spacer_worst)
                else:
                    _pair_spacer, _pair_spacer_worst = "", None
                _built, _problem = fuse(_d5, _d3, spacer=_pair_spacer)
                if _problem is not None:
                    continue
                # Same deterministic per-pair rotation as the FULL SWEEP cell above
                # -- never randomness (CLAUDE.md sec 6). _pick_fake_seq is reused
                # as-is, already defined by that earlier cell's own execution.
                _offset = (_i5 + _i3) % len(_mega_pool)
                _fake1 = pick_fake_design(
                    _d5, _d3, {_d5["rank"], _d3["rank"]}, _mega_pool,
                    start_offset=_offset,
                )
                _fake2 = (
                    pick_fake_design(
                        _d5, _d3, {_d5["rank"], _d3["rank"], _fake1["rank"]}, _mega_pool,
                        start_offset=_offset + 1,
                    )
                    if _fake1 is not None else None
                )
                if _fake1 is None or _fake2 is None:
                    _mega_skipped_no_fake += 1
                    continue
                _key = f"{_d5['rank']}+{_d3['rank']}"
                _mega_built_by_key[_key] = (_d5, _d3, _built, _pair_spacer, _pair_spacer_worst)
                _mega_jobs.append(FakeSweepJob(
                    key=_key,
                    fused=_built["fused"],
                    trigger_a=_d5["trigger"],
                    trigger_b=_d3["trigger"],
                    fake_a=_fake1["trigger"],
                    fake_b=_fake2["trigger"],
                    footprint_5p=_built["footprint_5p"],
                    footprint_3p=_built["footprint_3p"],
                    temperature=TEMPERATURE_C,
                    n_samples=MEGA_N_SAMPLES,
                    threshold=MEGA_THRESHOLD,
                ))

        _run_jobs = _mega_jobs if MEGA_PAIR_LIMIT is None else _mega_jobs[:MEGA_PAIR_LIMIT]
        print(f"  [{_spacer_label:>9}] {len(_mega_jobs)} pairs eligible "
              f"({_mega_excluded} overlap-excluded, "
              f"{_mega_skipped_no_fake} no-fake-available), "
              f"running {len(_run_jobs)}")

        _t0 = _time_mega.time()
        _mega_by_key = run_parallel_fake_sweep(_run_jobs, max_workers=MEGA_MAX_WORKERS)
        _elapsed = _time_mega.time() - _t0
        print(f"  [{_spacer_label:>9}] done in {_elapsed:.0f}s "
              f"({_elapsed / max(len(_run_jobs), 1):.2f}s/pair)")

        _combo_results = []
        for _key, (_d5, _d3, _built, _pair_spacer, _pair_spacer_worst) in _mega_built_by_key.items():
            if _key not in _mega_by_key:
                continue
            _dg_row = measure(
                _built, _d5["trigger"], _d3["trigger"], folder,
                _d3["metrics"]["own_kozak_rc_score"], _d5["metrics"]["own_prefix_rc_score"],
            )
            _combo_results.append({
                "csv_label": _csv_label, "spacer_label": _spacer_label,
                "rank_5p": _d5["rank"], "rank_3p": _d3["rank"],
                # Only the one ranking scalar is kept from the dG dict -- the full
                # measure() output is a raw-value dict, not JSON-round-trip-critical
                # for this cache; the report only ever reads d_ee_worst_alt from it.
                "dG": {"d_ee_worst_alt": _dg_row["d_ee_worst_alt"]},
                "states": _mega_by_key[_key],
                # "" for no_spacer, this pair's own optimized spacer for spacer --
                # downstream cells (AND top-20 selection/enrichment) reuse THIS
                # value rather than re-deriving it, so the same molecule is what
                # gets refined/reported, never a second independent pick.
                "spacer_seq": _pair_spacer, "spacer_worst_run": _pair_spacer_worst,
            })
        _combo_results.sort(key=lambda r: -r["states"]["+A+B"]["sampled_both"])
        mega_sweep_results[(_csv_label, _spacer_label)] = _combo_results
        mega_sweep_meta[(_csv_label, _spacer_label)] = {
            "n_pairs": len(_combo_results), "n_skipped_no_fake": _mega_skipped_no_fake,
            "elapsed_s": _elapsed,
            "mean_spacer_worst_run": (
                sum(_mega_spacer_worst_runs) / len(_mega_spacer_worst_runs)
                if _mega_spacer_worst_runs else None
            ),
        }
        if _spacer_label == "spacer" and _mega_spacer_worst_runs:
            print(f"  [{_spacer_label:>9}] per-pair spacer optimization: mean worst "
                  f"complementary run {sum(_mega_spacer_worst_runs) / len(_mega_spacer_worst_runs):.1f}nt "
                  f"over {len(_mega_spacer_worst_runs)} pairs (lower is better; pool of "
                  f"{SPACER_CANDIDATE_POOL_SIZE} candidates, {SPACER_MIN_LENGTH}-{SPACER_MAX_LENGTH}nt)")

_mega_total_elapsed = _time_mega.time() - _mega_started_all
print(f"\nmega sweep done in {_mega_total_elapsed:.0f}s "
      f"({_mega_total_elapsed / 60:.1f} min) total")


print(f"\n{'csv':>22}  {'spacer':>9}  {'n pairs':>7}  {'best +A+B both':>15}  "
      f"{'alone A open':>12} {'alone B open':>12}  (at best-both pair)")
for _csv_label in THREE_CSV_LABELS:
    for _spacer_label in MEGA_SPACER_LABELS:
        _rows = mega_sweep_results[(_csv_label, _spacer_label)]
        if not _rows:
            print(f"{_csv_label:>22}  {_spacer_label:>9}  {'0':>7}  "
                  f"{'--':>15}  {'--':>12} {'--':>12}")
            continue
        _best = _rows[0]
        _alone = _best["states"]["alone"]
        print(f"{_csv_label:>22}  {_spacer_label:>9}  {len(_rows):>7}  "
              f"{_best['states']['+A+B']['sampled_both']:>14.1%}  "
              f"{_alone['sampled_a_open']:>11.1%} {_alone['sampled_b_open']:>12.1%}  "
              f"(pair {_best['rank_5p']}+{_best['rank_3p']})")



===== sele v4 (15/18/21nt, 804b5213): top 20 designs (fake-informed score) (ranks ['143', '84', '37', '89', '30', '130', '103', '65', '68', '44', '150', '5', '49', '87', '86', '13', '85', '123', '42', '93']) =====
  [no_spacer] 354 pairs eligible (26 overlap-excluded, 0 no-fake-available), running 354


[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm 2
[ERROR]   backtracking failed for qm 2
[ERROR]   backtracking failed for qm 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_re

  [no_spacer] done in 59s (0.17s/pair)
  [   spacer] 354 pairs eligible (26 overlap-excluded, 0 no-fake-available), running 354


[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm 2
[ERROR]   backtracking failed for qm 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking failed for qm2_real 2
[ERROR]   backtracking

KeyboardInterrupt: 

## Final combined report — top 20 single-switch, top 20 AND, ranked by open minus leak

Combines Part 1's three-CSV single-switch comparison and the mega AND fake-leakage
sweep above into ONE report. Both top-20 lists use the same idea: rank by real signal
minus real leakage, never by signal alone (a candidate that is "open" only because it
is chronically leaky is not a good design — this notebook's own spacer experiment
found exactly that failure mode, and it would win a signal-only ranking easily).


In [ ]:
# ==================================================================================
# SINGLE-SWITCH TOP-20 (for the report) -- flattens `single_scored_by_csv` (built by
# the SINGLE-SWITCH SCORING cell above) across all three CSVs into one globally-ranked
# list of 20. Same score as the AND pool selection uses (on_eff/off_eff, fake-informed,
# worst-of-two) -- one scoring function for this whole notebook's single-switch
# ranking, not a second one just for the report's top 20.
# ==================================================================================
single_top20 = []
for _csv_label in THREE_CSV_LABELS:
    for _e in single_scored_by_csv[_csv_label]:
        _row, _fc = _e["row"], _e["fake_check"]
        single_top20.append({
            "csv_label": _csv_label, "rank": _row["design"]["rank"],
            "open": _e["on_eff"], "leak": _e["off_eff"], "score": _e["score"],
            "d_ee": _row["d_ee"], "trigger_len": len(_row["design"]["trigger"]),
            # Kept for the final report: full sequence, domain breakdown, and the real
            # true+wrong-trigger ensemble check (NUPACK, Boltzmann-sampled). Never None
            # here -- only candidates with every fake-check number measured made it
            # into single_scored_by_csv at all (see the scoring cell above).
            "switch": _row["design"]["switch"], "domains": _row["design"]["domains"],
            "trigger": _row["design"]["trigger"],
            # Position on the target transcript, read straight off the design row (the
            # CSV's own trigger_start_pos) -- feeds the report's trigger-position map.
            "trigger_start": _row["design"].get("trigger_start"),
            "alone_real_paired_fraction": _row.get("alone_real_paired_fraction"),
            "sample_coverage": _row.get("sample_coverage"),
            "effective_structures": _row.get("effective_structures"),
            "fake_check": _fc,
        })

single_top20.sort(key=lambda r: -r["score"])
single_top20 = single_top20[:20]

print(f"{'#':>3}  {'csv':>22}  {'rank':>5}  {'ON eff':>7}  {'OFF eff':>8}  {'score':>7}")
for _i, _r in enumerate(single_top20, start=1):
    print(f"{_i:>3}  {_r['csv_label']:>22}  {_r['rank']:>5}  {_r['open']:>7.1%}  "
          f"{_r['leak']:>8.1%}  {_r['score']:>+7.1%}")

In [ ]:
# ==================================================================================
# AND TOP-20 -- flattens `mega_sweep_results` (built by the MEGA FAKE-LEAKAGE SWEEP
# cell above) into a ranked list. Two lists are built with the exact same scoring/
# enrichment logic (one function, called twice -- not two implementations):
#   - `and_top20`: combined across all three CSVs AND both spacer configs. In
#     practice the spacer variant's higher scores fill every slot of this list, so it
#     never actually shows a no-spacer pair.
#   - `and_top20_no_spacer`: the same ranking, restricted up front to spacer_label ==
#     "no_spacer" only, so the no-spacer variant's own best pairs are visible on their
#     own terms instead of being crowded out.
#
# SCORE (`_and_score`, the ONE function every ranking pass in this notebook's AND side
# calls -- the initial 20x20 screen here, and the post-refinement re-sort in the next
# cell) is now fake-AND-single-trigger-informed, not just the "+A+B" vs "alone" pair
# the previous version of this cell used:
#
#   open_eff = min(+A+B both, +A+B+fake1+fake2 both) -- the worse of "does giving it
#     both real triggers actually activate it" and "does that still hold with two
#     off-targets also present".
#   leak_eff = max(both, across EVERY state that should NOT read as full activation:
#     alone, +fake1, +fake2, +fake1+fake2, +A, +B, +A+fake1, +B+fake2) -- the worst of
#     eight distinct ways this pair could falsely look "on". Critically this now
#     includes +A and +B (one real trigger, nothing else) -- the exact AND-vs-OR
#     question this whole notebook started from ("why does trigger B always win"),
#     which the previous score never actually checked: it only ever looked at the
#     alone state's MARGINAL per-footprint openness, never whether a single real
#     trigger already drives the JOINT "both open" reading on its own.
#
# score = open_eff - leak_eff. Same worst-of-many principle as always in this
# notebook -- a design cannot hide one bad state behind the others being clean.
#
# CAVEAT this cell's own ranking still carries: the initial 20x20 screen (this cell)
# uses whatever fake draw the mega sweep took for THIS pair (one draw for the full
# grid, for cost reasons -- documented on the mega sweep cell above). The next cell
# refines the six fake-involving states to a 10-fake mean and RE-SCORES/RE-SORTS with
# the same `_and_score` afterward -- but only for the 20 pairs THIS screen already
# selected. A pair just outside the top 20 on the noisier 1-draw score is never
# reconsidered even if a 10-fake mean would have ranked it higher.
# ==================================================================================
_and_pool_cache = {}


def _and_pool(csv_label):
    """Rank->design lookup for one CSV's pool -- the SAME fake-informed top-20
    (`single_top20_by_csv`, built by the SINGLE-SWITCH SCORING cell above) the mega
    sweep itself was built from, not a separate `load_ranked_designs(...)` call --
    the AND side's pool and the mega sweep's pool must be identical designs, and
    fetching them from one shared source is what guarantees that rather than assuming
    two calls with the same `limit` happen to agree."""
    if csv_label not in _and_pool_cache:
        _and_pool_cache[csv_label] = {d["rank"]: d for d in single_top20_by_csv[csv_label]}
    return _and_pool_cache[csv_label]


# States that should NOT read as full ("both") activation -- everything except the two
# genuinely-on states. Shared by the initial screen here and the post-refinement
# re-score in the next cell, so both passes agree on what "leak" means.
_AND_ILLEGITIMATE_STATES = ("alone", "+fake1", "+fake2", "+fake1+fake2", "+A", "+B", "+A+fake1", "+B+fake2")
_AND_LEGITIMATE_STATES = ("+A+B", "+A+B+fake1+fake2")


def _and_score(states):
    """(open_eff, leak_eff, score) from a pair's full 10-state dict -- see this cell's
    own docstring above for what each half means and why. The ONE scoring function
    every AND ranking pass in this notebook calls."""
    open_eff = min(states[s]["sampled_both"] for s in _AND_LEGITIMATE_STATES)
    leak_eff = max(states[s]["sampled_both"] for s in _AND_ILLEGITIMATE_STATES)
    return open_eff, leak_eff, open_eff - leak_eff


def _select_and_top20(spacer_filter=None, n=20):
    """Ranks `mega_sweep_results` by `_and_score`, optionally restricted to one spacer
    label first, then re-fetches each selected pair's own two designs and re-runs
    fuse() (cheap: no folding, just string/domain relocation) to attach the full fused
    sequence + domain breakdown for the final report -- mega_sweep_results only ever
    kept the scalar sampled fractions and the dG summary, not the built molecule
    itself."""
    rows = []
    for (_csv_label, _spacer_label), _rows in mega_sweep_results.items():
        if spacer_filter is not None and _spacer_label != spacer_filter:
            continue
        for _row in _rows:
            _st = _row["states"]
            _open, _leak, _ = _and_score(_st)
            rows.append({
                "csv_label": _csv_label, "spacer_label": _spacer_label,
                "rank_5p": _row["rank_5p"], "rank_3p": _row["rank_3p"],
                "open": _open, "leak": _leak, "score": _open - _leak,
                "d_ee_worst_alt": _row["dG"]["d_ee_worst_alt"], "states": _st,
                # This pair's own per-pair-optimized spacer (empty string for
                # no_spacer) -- the SAME value the mega sweep actually folded, not
                # SPACER_SEQ (the old fixed global, now only fuse()'s fallback
                # default). Re-deriving it here instead of reusing the stored value
                # would risk a different pick if SPACER_CANDIDATES ever changed
                # between cells.
                "spacer_seq": _row.get("spacer_seq", ""),
                "spacer_worst_run": _row.get("spacer_worst_run"),
            })

    rows.sort(key=lambda r: -r["score"])
    rows = rows[:n]

    for _r in rows:
        _by_rank = _and_pool(_r["csv_label"])
        _d5, _d3 = _by_rank[_r["rank_5p"]], _by_rank[_r["rank_3p"]]
        _built, _problem = fuse(_d5, _d3, spacer=_r["spacer_seq"])
        assert _problem is None, _problem
        _r["built"] = _built
        _r["trigger_a"] = _d5["trigger"]
        _r["trigger_b"] = _d3["trigger"]
    return rows


def _print_and_top(label, rows):
    print(f"\n{label}")
    print(f"{'#':>3}  {'csv':>22}  {'spacer':>9}  {'pair':>8}  {'open':>7}  {'leak':>7}  {'score':>7}")
    for _i, _r in enumerate(rows, start=1):
        print(f"{_i:>3}  {_r['csv_label']:>22}  {_r['spacer_label']:>9}  "
              f"{_r['rank_5p']}+{_r['rank_3p']:<5}  {_r['open']:>7.1%}  "
              f"{_r['leak']:>7.1%}  {_r['score']:>+7.1%}")


and_top20 = _select_and_top20(spacer_filter=None)
_print_and_top("AND top 20 (combined, both spacer configs)", and_top20)

and_top20_no_spacer = _select_and_top20(spacer_filter="no_spacer")
_print_and_top("AND top 20 (no-spacer only)", and_top20_no_spacer)

and_top20_spacer = _select_and_top20(spacer_filter="spacer")
_print_and_top("AND top 20 (spacer variant only)", and_top20_spacer)

In [ ]:
# ==================================================================================
# AND TOP-20 -- refine the six fake-INVOLVING states with 10 distinct, non-overlapping
# fake triggers each, averaged. The mega sweep above (all 2032 pairs) used exactly one
# rotated (fake1, fake2) draw per pair -- a reasonable choice for a pool-wide screen,
# but a single draw's idiosyncrasy is exactly the kind of thing this notebook found and
# fixed once already (the fake-rotation bug). For the pairs that actually make the
# final report (both the combined top-20 and the no-spacer-only top-20), this cell
# reruns just the fake-involving states
# (+fake1/+fake2/+fake1+fake2/+A+fake1/+B+fake2/+A+B+fake1+fake2) against
# AND_TOP20_N_FAKES independent (fake1, fake2) draws and averages them -- a properly
# robust number instead of one draw. `alone`/`+A`/`+B`/`+A+B` do not depend on any fake
# choice at all and are left untouched. `open`/`leak`/`score` ARE recomputed afterward,
# with `_and_score` (defined in the AND TOP-20 selection cell above, reused here rather
# than reimplemented) fed the now-refined states -- since the score depends on
# +A+fake1/+B+fake2/+fake1/+fake2/+fake1+fake2/+A+B+fake1+fake2, all six of which just
# changed from a 1-draw estimate to a 10-draw mean, the ranking WITHIN this already-
# selected top-20 can and does shift. This does not reconsider which 20 pairs made the
# cut in the first place (see the selection cell's own caveat on that).
#
# One function, called once per top-20 list, so the combined and no-spacer-only
# results are computed by the exact same logic rather than two copies of it.
#
# HOW TO RE-RUN THIS YOURSELF: run every cell through the AND TOP-20 cell above first
# (needs `and_top20`, `and_top20_no_spacer`, `_and_pool`, `folder`, `TEMPERATURE_C`,
# `MIN_TRIGGER_SEPARATION_NT`, `MEGA_N_SAMPLES`, `MEGA_THRESHOLD`, `MEGA_MAX_WORKERS`,
# `FakeSweepJob`, `pick_fake_design`, `run_parallel_fake_sweep` -- all already
# imported/defined above). Adjust the parameter below, then run just this cell.
# ==================================================================================
AND_TOP20_N_FAKES = 10

_FAKE_INVOLVING_STATES = {
    "+fake1", "+fake2", "+fake1+fake2", "+A+fake1", "+B+fake2", "+A+B+fake1+fake2",
}


def _refine_and_fake_states(entries, label):
    _refine_jobs_by_pair = {}
    for _r in entries:
        _built = _r["built"]
        _fp5, _fp3 = _built["footprint_5p"], _built["footprint_3p"]
        _pool_by_rank = _and_pool(_r["csv_label"])
        _pool_list = list(_pool_by_rank.values())
        _d5, _d3 = _pool_by_rank[_r["rank_5p"]], _pool_by_rank[_r["rank_3p"]]

        _jobs = []
        for _k in range(AND_TOP20_N_FAKES):
            _fake1 = pick_fake_design(
                _d5, _d3, {_d5["rank"], _d3["rank"]}, _pool_list,
                start_offset=_k,
            )
            _fake2 = (
                pick_fake_design(
                    _d5, _d3, {_d5["rank"], _d3["rank"], _fake1["rank"]}, _pool_list,
                    start_offset=_k + 1,
                )
                if _fake1 is not None else None
            )
            if _fake1 is None or _fake2 is None:
                break
            _key = f"{_r['csv_label']}|{_r['spacer_label']}|{_r['rank_5p']}+{_r['rank_3p']}|fake{_k}"
            _jobs.append(FakeSweepJob(
                key=_key, fused=_built["fused"], trigger_a=_r["trigger_a"], trigger_b=_r["trigger_b"],
                fake_a=_fake1["trigger"], fake_b=_fake2["trigger"],
                footprint_5p=_fp5, footprint_3p=_fp3, temperature=TEMPERATURE_C,
                n_samples=MEGA_N_SAMPLES, threshold=MEGA_THRESHOLD,
            ))
        _refine_jobs_by_pair[id(_r)] = _jobs

    _all_refine_jobs = [j for jobs in _refine_jobs_by_pair.values() for j in jobs]
    print(f"running {len(_all_refine_jobs)} jobs ({AND_TOP20_N_FAKES} fakes x {len(entries)} pairs, "
          f"6 fake-involving states each computed inside every job) -- {label}")
    _refine_started = time.time()
    _refine_by_key = run_parallel_fake_sweep(_all_refine_jobs, max_workers=MEGA_MAX_WORKERS)
    _refine_elapsed = time.time() - _refine_started
    print(f"done in {_refine_elapsed:.0f}s ({_refine_elapsed / max(len(_all_refine_jobs), 1):.2f}s/job)")

    for _r in entries:
        _jobs = _refine_jobs_by_pair[id(_r)]
        _per_state = {s: {"a": [], "b": [], "both": []} for s in _FAKE_INVOLVING_STATES}
        for _job in _jobs:
            _res = _refine_by_key.get(_job.key)
            if _res is None:
                continue
            for _s in _FAKE_INVOLVING_STATES:
                _per_state[_s]["a"].append(_res[_s]["sampled_a_open"])
                _per_state[_s]["b"].append(_res[_s]["sampled_b_open"])
                _per_state[_s]["both"].append(_res[_s]["sampled_both"])
        for _s in _FAKE_INVOLVING_STATES:
            _a, _b, _both = _per_state[_s]["a"], _per_state[_s]["b"], _per_state[_s]["both"]
            _r["states"][_s] = {
                "sampled_a_open": sum(_a) / len(_a) if _a else None,
                "sampled_b_open": sum(_b) / len(_b) if _b else None,
                "sampled_both": sum(_both) / len(_both) if _both else None,
                "n_fakes_checked": len(_a),
            }

    # Re-score with the now-refined (10-fake mean) states, using the SAME _and_score
    # the selection cell above used to pick these 20 pairs in the first place -- one
    # scoring function, called again on better data, not a second formula. Skip (keep
    # the pre-refinement score) only for the rare entry where a fake draw ran out
    # entirely (n_fakes_checked == 0 for some state -- sampled_both is None then) --
    # CLAUDE.md sec 3: no default standing in for a measurement not taken.
    _n_rescored = 0
    for _r in entries:
        if any(_r["states"][_s]["sampled_both"] is None for _s in _FAKE_INVOLVING_STATES):
            continue
        _open, _leak, _score = _and_score(_r["states"])
        _r["open"], _r["leak"], _r["score"] = _open, _leak, _score
        _n_rescored += 1
    entries.sort(key=lambda r: -r["score"])

    print(f"refined {len(entries)} AND entries' fake-involving states "
          f"(mean over up to {AND_TOP20_N_FAKES} distinct non-overlapping fakes each) -- {label}; "
          f"re-scored and re-sorted {_n_rescored}/{len(entries)} entries with the refined data")


_refine_and_fake_states(and_top20, "combined top 20")
_refine_and_fake_states(and_top20_no_spacer, "no-spacer top 20")
_refine_and_fake_states(and_top20_spacer, "spacer-only top 20")

In [ ]:
# ==================================================================================
# AND TOP-20 -- CONSTRAINED-BIND CHECK: if one role is forced into its REAL bound
# duplex (the exact base pairs it forms with its own trigger in isolation -- read off
# an actual MFE fold, not assumed), does the OTHER role still bind its own trigger,
# for real, in the remainder of the structure? Run with the forced role's trigger
# alone present, and separately with one off-target fake also present -- so each
# direction gets both a clean and a noisy version, same as every other fake-informed
# check in this notebook.
#
# THIS IS A DIFFERENT QUESTION FROM `sampled_both` (the real unconstrained joint
# probability already reported elsewhere): forcing one duplex removes every
# competing conformation for that region, so it answers "CAN the other role still
# bind GIVEN this one is genuinely bound" rather than "does the real, unconstrained
# system actually choose to settle there on its own". Both numbers are kept,
# side by side -- this cell adds to the existing states, it does not replace them.
#
# Four new per-pair metrics, each a dict with "check_bound_frac" (the real question),
# "n_forced_pairs" (how much of the forcing role's footprint was actually forced --
# a low number here means this design's own real binding was weak or partial, not a
# cleanly-forced 100% duplex, and the check's own meaning weakens accordingly;
# CLAUDE.md sec 3 -- never silently treated as if it were a clean full duplex), and
# the usual sample_coverage/effective_structures pair:
#   "A_bound_check_B"       -- A forced bound (real duplex), is B also bound? clean.
#   "B_bound_check_A"       -- B forced bound (real duplex), is A also bound? clean.
#   "A_bound_check_B_fake"  -- same as above, PLUS one off-target fake present too
#                              (mean over up to CONSTRAINED_N_FAKES distinct fakes).
#   "B_bound_check_A_fake"  -- symmetric with-fake version.
#
# NOT YET RUN: this cell is new. Smoke-tested standalone against one real pair
# outside the notebook (zero crashes, plausible 0.0/1.0-range outputs, forced-pair
# counts > 0) before being wired in here -- but never yet run end-to-end inside this
# notebook's own pipeline. Treat the first real run's numbers as provisional until
# spot-checked.
# ==================================================================================
import sys as _sys_cbind
from pathlib import Path as _Path_cbind

_here_cbind = _Path_cbind.cwd()
for _base in (_here_cbind, *_here_cbind.parents):
    if (_base / "_joint_state_parallel.py").exists():
        if str(_base) not in _sys_cbind.path:
            _sys_cbind.path.insert(0, str(_base))
        break
from _joint_state_parallel import ConstrainedBindJob, run_parallel_constrained_bind
import time as _time_cbind

CONSTRAINED_N_SAMPLES = 1200
CONSTRAINED_THRESHOLD = 0.80
CONSTRAINED_N_FAKES = 10  # distinct, non-overlapping-by-sequence fakes averaged for the "_fake" variants
CONSTRAINED_MAX_WORKERS = 6


def _add_constrained_bind_checks(entries, label):
    """Two passes, ONE process pool: build every job for every pair up front, run them
    all in a single ``run_parallel_constrained_bind`` call, then distribute the results
    back onto their own pair. The first version of this cell called the pool twice per
    pair (120 spin-ups for 60 pairs) and its "clean" call held only 2 jobs, so six
    workers sat mostly idle; cell 21's refinement pass already used the batch-everything
    shape, and this now matches it. Job keys are namespaced by the pair's index in
    ``entries`` so one flat result dict can serve every pair unambiguously.
    """
    _started = _time_cbind.time()
    _jobs = []
    _fakes_by_pair = {}

    for _pi, _r in enumerate(entries):
        _built = _r["built"]
        _fused = _built["fused"]
        _fp5, _fp3 = _built["footprint_5p"], _built["footprint_3p"]
        _pool_by_rank = _and_pool(_r["csv_label"])
        _pool_list = list(_pool_by_rank.values())
        _d5, _d3 = _pool_by_rank[_r["rank_5p"]], _pool_by_rank[_r["rank_3p"]]

        def _job(_key, _bound_trig, _check_trig, _fp_bound, _fp_check, _fake=""):
            return ConstrainedBindJob(
                key=f"{_pi}|{_key}", fused=_fused, trigger_bound=_bound_trig,
                trigger_check=_check_trig, footprint_bound=_fp_bound,
                footprint_check=_fp_check, fake=_fake, temperature=TEMPERATURE_C,
                n_samples=CONSTRAINED_N_SAMPLES, threshold=CONSTRAINED_THRESHOLD,
            )

        # ---- clean (no fake), one each direction ----
        _jobs.append(_job("A_bound_check_B", _r["trigger_a"], _r["trigger_b"], _fp5, _fp3))
        _jobs.append(_job("B_bound_check_A", _r["trigger_b"], _r["trigger_a"], _fp3, _fp5))

        # ---- the fakes for this pair, picked once and reused by every fake variant ----
        _picked_ranks = {_d5["rank"], _d3["rank"]}
        _fakes = []
        for _k in range(CONSTRAINED_N_FAKES):
            _wd = pick_fake_design(_d5, _d3, set(_picked_ranks), _pool_list, start_offset=_k)
            if _wd is None:
                break
            _picked_ranks.add(_wd["rank"])
            _fakes.append(_wd)
        _fakes_by_pair[_pi] = _fakes

        # Two kinds per direction: the real other-trigger PLUS a fake (noise on top of
        # the real signal), and the fake ALONE as the only other strand (can an
        # off-target open the other footprint by itself, given this role is genuinely
        # bound -- the constrained-check analog of the "fake only" leak numbers
        # elsewhere in this notebook). Both reuse the SAME picked `_fakes`: one
        # fake-selection pass, not two independent draws that could disagree.
        for _k, _wd in enumerate(_fakes):
            _jobs.append(_job(f"A_bound_check_B_fake{_k}", _r["trigger_a"], _r["trigger_b"],
                              _fp5, _fp3, _fake=_wd["trigger"]))
            _jobs.append(_job(f"B_bound_check_A_fake{_k}", _r["trigger_b"], _r["trigger_a"],
                              _fp3, _fp5, _fake=_wd["trigger"]))
            # fake-ONLY: the fake stands in as `trigger_check` itself, no real
            # other-trigger strand at all -- a 3-strand complex, not 4.
            _jobs.append(_job(f"A_bound_check_fakeonly{_k}", _r["trigger_a"], _wd["trigger"],
                              _fp5, _fp3))
            _jobs.append(_job(f"B_bound_check_fakeonly{_k}", _r["trigger_b"], _wd["trigger"],
                              _fp3, _fp5))

    _results = run_parallel_constrained_bind(_jobs, max_workers=CONSTRAINED_MAX_WORKERS)

    for _pi, _r in enumerate(entries):
        _fakes = _fakes_by_pair[_pi]
        _clean_a = _results[f"{_pi}|A_bound_check_B"]
        _clean_b = _results[f"{_pi}|B_bound_check_A"]

        def _mean_of(_prefix):
            vals = [
                _results[f"{_pi}|{_prefix}{_k}"]["check_bound_frac"]
                for _k in range(len(_fakes))
            ]
            return sum(vals) / len(vals) if vals else None

        _r["constrained"] = {
            "A_bound_check_B": _clean_a,
            "B_bound_check_A": _clean_b,
            "A_bound_check_B_fake": {
                "check_bound_frac": _mean_of("A_bound_check_B_fake"),
                "n_forced_pairs": _clean_a["n_forced_pairs"],
                "n_fakes_checked": len(_fakes),
            },
            "B_bound_check_A_fake": {
                "check_bound_frac": _mean_of("B_bound_check_A_fake"),
                "n_forced_pairs": _clean_b["n_forced_pairs"],
                "n_fakes_checked": len(_fakes),
            },
            "A_bound_check_fakeonly": {
                "check_bound_frac": _mean_of("A_bound_check_fakeonly"),
                "n_forced_pairs": _clean_a["n_forced_pairs"],
                "n_fakes_checked": len(_fakes),
            },
            "B_bound_check_fakeonly": {
                "check_bound_frac": _mean_of("B_bound_check_fakeonly"),
                "n_forced_pairs": _clean_b["n_forced_pairs"],
                "n_fakes_checked": len(_fakes),
            },
        }

    _elapsed = _time_cbind.time() - _started
    print(f"  {label}: {len(entries)} pairs, {len(_jobs)} constrained-bind jobs "
          f"in 1 pool call, {_elapsed:.0f}s")


_add_constrained_bind_checks(and_top20, "combined top 20")
_add_constrained_bind_checks(and_top20_no_spacer, "no-spacer top 20")
_add_constrained_bind_checks(and_top20_spacer, "spacer-only top 20")


In [ ]:
# ==================================================================================
# EXPORT RESULTS TO CSV -- explicit, opt-in snapshot of everything the report below
# is built from. Run this cell BY HAND whenever you want a durable copy of the
# current kernel's results (e.g. before touching report layout, or before risking a
# kernel restart) -- nothing upstream writes to disk on its own anymore (the disk
# caches were removed from cells 11/17; a restarted kernel now loses everything
# unless you've run this). Two flat CSVs, one row per candidate, every nested
# structure (domains, per-state fake-check numbers) JSON-encoded into its own
# column -- the same convention this notebook's own NucSyn CSV inputs already use
# for switch_domain_breakdown_json, not a new one. A shared `exported_at` column
# (plus the same timestamp in both filenames) says exactly when this snapshot was
# taken, so an old export can never be mistaken for a fresh one, and re-running this
# cell later never overwrites an earlier snapshot.
# ==================================================================================
import csv as _csv_export
import datetime as _dt_export

_export_repo_root = Path.cwd()
for _base in (_export_repo_root, *_export_repo_root.parents):
    if (_base / "pyproject.toml").exists():
        _export_repo_root = _base
        break

_export_ts = _dt_export.datetime.now().strftime("%Y%m%dT%H%M%S")
_export_dir = _export_repo_root / "var" / "reports"
_export_dir.mkdir(parents=True, exist_ok=True)


def _write_csv(path, rows, fieldnames):
    with path.open("w", newline="") as handle:
        writer = _csv_export.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f"wrote {len(rows)} rows -> {path}")


# ---- single-switch top 20 --------------------------------------------------------
_single_export_rows = [
    {
        "exported_at": _export_ts,
        "csv_label": _r["csv_label"], "rank": _r["rank"],
        "open": _r["open"], "leak": _r["leak"], "score": _r["score"],
        "d_ee": _r["d_ee"], "trigger_len": _r["trigger_len"],
        "trigger": _r["trigger"], "switch": _r["switch"],
        "alone_real_paired_fraction": _r["alone_real_paired_fraction"],
        "sample_coverage": _r["sample_coverage"],
        "effective_structures": _r["effective_structures"],
        "domains_json": json.dumps(_r["domains"]),
        "fake_check_json": json.dumps(_r["fake_check"]),
    }
    for _r in single_top20
]
_single_export_path = _export_dir / f"single_top20_{_export_ts}.csv"
_write_csv(_single_export_path, _single_export_rows, list(_single_export_rows[0].keys()))

# ---- AND top 20: combined + no-spacer-only in one file, `list_name` tells them
# apart (a pair can legitimately show up in both) ----------------------------------
_and_export_rows = [
    {
        "exported_at": _export_ts, "list_name": _list_name,
        "csv_label": _r["csv_label"], "spacer_label": _r["spacer_label"],
        "rank_5p": _r["rank_5p"], "rank_3p": _r["rank_3p"],
        "open": _r["open"], "leak": _r["leak"], "score": _r["score"],
        "d_ee_worst_alt": _r["d_ee_worst_alt"],
        "spacer_seq": _r["spacer_seq"], "spacer_worst_run": _r["spacer_worst_run"],
        "trigger_a": _r["trigger_a"], "trigger_b": _r["trigger_b"],
        "fused_switch": _r["built"]["fused"],
        "domains_5p_json": json.dumps(_r["built"]["domains_5p"]),
        "domains_3p_json": json.dumps(_r["built"]["domains_3p"]),
        "states_json": json.dumps(_r["states"]),
        "constrained_json": json.dumps(_r.get("constrained")),
    }
    for _list_name, _rows in (
        ("combined", and_top20), ("no_spacer_only", and_top20_no_spacer),
        ("spacer_only", and_top20_spacer),
    )
    for _r in _rows
]
_and_export_path = _export_dir / f"and_top20_{_export_ts}.csv"
_write_csv(_and_export_path, _and_export_rows, list(_and_export_rows[0].keys()))

# ---- full state snapshot (JSON) -- rebuild ANY report later without re-running ----
# The two CSVs above are for reading/pivoting by hand. This snapshot is for the
# notebook itself: it holds exactly the variables the report cells consume, so the
# "LOAD SNAPSHOT" cell below can restore them into a fresh kernel and go straight to
# rendering -- skipping the ~10 min single-switch cell, the mega sweep, the AND
# selection/refinement and the constrained-bind pass entirely.
#
# Images are deliberately NOT stored: they are base64 PNGs (tens of MB across ~220
# figures) and the images cell rebuilds them from the sequences in seconds, so keeping
# them would bloat every snapshot to save almost nothing. The loader re-runs that cell.
#
# `single_top20_by_csv` is included because `_and_pool()` (used by the AND cards and the
# trigger-position maps) is built from it -- without it a restored session could render
# the tables but not the maps.
_IMAGE_KEYS = ("structure_png", "matrix_png", "layout_png")


def _strip_images(rows):
    return [{k: v for k, v in r.items() if k not in _IMAGE_KEYS} for r in rows]


_snapshot = {
    "exported_at": _export_ts,
    "temperature_c": TEMPERATURE_C,
    "transcript_length": SELE_TRANSCRIPT_LENGTH,
    "three_csv_labels": list(THREE_CSV_LABELS),
    "three_csvs": list(THREE_CSVS),
    "single_top20": _strip_images(single_top20),
    "and_top20": _strip_images(and_top20),
    "and_top20_no_spacer": _strip_images(and_top20_no_spacer),
    "and_top20_spacer": _strip_images(and_top20_spacer),
    "single_top20_by_csv": {k: v for k, v in single_top20_by_csv.items()},
}
_snapshot_path = _export_dir / f"report_snapshot_{_export_ts}.json"
_snapshot_path.write_text(json.dumps(_snapshot))
print(f"wrote state snapshot -> {_snapshot_path} "
      f"({_snapshot_path.stat().st_size / 1e6:.1f} MB)")

print(f"\nsnapshot timestamp: {_export_ts}")
print("to rebuild a report later without re-running the pipeline: run the setup cells "
      "(through the plotting/CSS helpers), then the LOAD SNAPSHOT cell below, then the "
      "images + report cells.")


In [ ]:
# ==================================================================================
# LOAD SNAPSHOT -- restore a previous run's results instead of recomputing them.
#
# RUN THIS *INSTEAD OF* the heavy cells (the top-150 single-switch comparison, the
# mega fake-leakage sweep, the AND selection/refinement, the constrained-bind pass)
# when all you want is a different report off data you already computed. Order:
#
#   1. every setup/helper cell above Part 1, i.e. the path bootstrap, the inputs cell,
#      the loader cell, the plotting-helpers cell and the report-print-helpers cell
#   2. the "PART 1 -- SINGLE SWITCH" cell that builds `folder` (the one FoldEngine) and
#      defines measure_single_switch() -- it no longer measures anything by itself, so
#      it costs nothing, but the images cell needs `folder` and will NameError without it
#   3. the fuse()/measure() definition cells from Part 2 (definitions only, no sweep)
#   4. the shared palette/CSS cell
#   5. THIS cell, with SNAPSHOT_TO_LOAD pointed at the file you want
#   6. the images cell, then the renderer + write-HTML/PDF/ready-to-publish cells
#
# Verified end to end in exactly that order against a real snapshot: the heavy cells
# (top-150 comparison, mega sweep, AND selection/refinement, constrained-bind) never
# execute and the report still renders.
#
# Leave SNAPSHOT_TO_LOAD = None for a normal full run -- then this cell is a no-op and
# does not touch anything the pipeline above it just computed. That default matters:
# a "Run All" must never silently replace a fresh computation with stale saved data,
# which is the whole failure mode the disk caches were removed for.
# ==================================================================================
SNAPSHOT_TO_LOAD = None   # e.g. "report_snapshot_20261001T143000.json", or an absolute path

if SNAPSHOT_TO_LOAD is None:
    print("SNAPSHOT_TO_LOAD is None -- no-op (normal full run; nothing restored).")
else:
    _snap_root = Path.cwd()
    for _base in (_snap_root, *_snap_root.parents):
        if (_base / "pyproject.toml").exists():
            _snap_root = _base / "var" / "reports"
            break
    _snap_path = Path(SNAPSHOT_TO_LOAD)
    if not _snap_path.is_absolute():
        _snap_path = _snap_root / _snap_path
    if not _snap_path.exists():
        _available = sorted(p.name for p in _snap_root.glob("report_snapshot_*.json"))
        raise FileNotFoundError(
            f"{_snap_path} does not exist. Snapshots available in {_snap_root}: "
            f"{_available or 'none -- run the export cell after a full run first'}"
        )

    _snap = json.loads(_snap_path.read_text())

    # Restored verbatim under the exact names the report cells read, so nothing
    # downstream needs to know whether it is running on fresh or restored data.
    single_top20 = _snap["single_top20"]
    and_top20 = _snap["and_top20"]
    and_top20_no_spacer = _snap["and_top20_no_spacer"]
    and_top20_spacer = _snap["and_top20_spacer"]
    single_top20_by_csv = _snap["single_top20_by_csv"]
    THREE_CSV_LABELS = tuple(_snap["three_csv_labels"])
    THREE_CSVS = tuple(_snap["three_csvs"])

    # `_and_pool` memoises per csv_label and is normally defined by the AND selection
    # cell, which a snapshot-only session never runs -- so it is (re)defined here over
    # the restored pool. Same lookup, same single source (single_top20_by_csv), so the
    # AND cards and trigger maps behave identically either way.
    _and_pool_cache = {}

    def _and_pool(csv_label):
        if csv_label not in _and_pool_cache:
            _and_pool_cache[csv_label] = {d["rank"]: d for d in single_top20_by_csv[csv_label]}
        return _and_pool_cache[csv_label]

    # JSON has no tuples: every (start, end) span came back as a 2-element list. The
    # report/plot code only ever indexes [0]/[1] or unpacks, which works the same for a
    # list -- but fuse()'s own verification and anything doing dict lookups keyed by a
    # span would not, so spans are put back to tuples rather than left half-converted.
    def _spans_to_tuples(obj):
        if isinstance(obj, dict):
            return {
                k: (tuple(v) if isinstance(v, list) and len(v) == 2
                    and all(isinstance(x, int) for x in v) else _spans_to_tuples(v))
                for k, v in obj.items()
            }
        if isinstance(obj, list):
            return [_spans_to_tuples(v) for v in obj]
        return obj

    single_top20 = _spans_to_tuples(single_top20)
    and_top20 = _spans_to_tuples(and_top20)
    and_top20_no_spacer = _spans_to_tuples(and_top20_no_spacer)
    and_top20_spacer = _spans_to_tuples(and_top20_spacer)
    single_top20_by_csv = _spans_to_tuples(single_top20_by_csv)

    print(f"restored snapshot {_snap_path.name} (exported {_snap['exported_at']}, "
          f"{_snap['temperature_c']:.1f} C)")
    print(f"  {len(single_top20)} single-switch, {len(and_top20)} AND (combined), "
          f"{len(and_top20_no_spacer)} AND (no-spacer), {len(and_top20_spacer)} AND (spacer)")
    print(f"  source CSV label(s): {', '.join(THREE_CSV_LABELS)}")
    print("  images are NOT in the snapshot -- run the images cell next to rebuild them, "
          "then the report cells.")


In [ ]:
# ==================================================================================
# FINAL REPORT -- images per entry (both top-20 lists): full sequence is already on
# each row (added by the two selection cells above); this cell adds a real 2D
# structure plot (`_structure_png`, already defined above -- FoldEngine.layout_coordinates,
# the same naview layout RNAplot/forna use) and a real base-pair-probability heatmap
# (new here -- the one plot type this notebook did not already have a function for)
# for every entry, plus the AND block-diagram image (`render_layout_bar_png`, already
# defined above) for every AND entry. Single-switch block diagram is text (domain name
# + length), not an image -- there is no two-half layout to draw for one hairpin, and a
# labeled list already says everything a bar would.
#
# Nothing here re-folds anything new: `folder.mfe`/`folder.base_pair_probabilities` are
# cached per sequence, and every sequence touched here was already folded once by
# measure()/measure_single_switch() earlier in this notebook's run, so this is cache
# hits, not fresh folding.
# ==================================================================================
import numpy as _np_final


def _bpp_heatmap_png(matrix, title, spans=(), figsize=(3.3, 3.0)):
    """A real base-pair-probability matrix, rendered as a heatmap -- `spans` (a list of
    (start, end, color) tuples) draws a colored outline box around a footprint region,
    the same visual language `_structure_png`'s ring outlines use."""
    arr = _np_final.array(matrix)
    fig, ax = plt.subplots(figsize=figsize, dpi=130)
    ax.imshow(arr, cmap="magma", vmin=0.0, vmax=1.0, origin="upper")
    for start, end, color in spans:
        ax.add_patch(plt.Rectangle((start - 0.5, start - 0.5), end - start, end - start,
                                    fill=False, edgecolor=color, linewidth=1.4))
    ax.set_title(title, fontsize=9, fontweight="bold")
    ax.set_xticks([])
    ax.set_yticks([])
    plt.tight_layout()
    return _png_from_fig(fig)


# ---- single-switch entries ----
# The 2D structure/bpp-matrix PLOTS render the switch through the end of Kozak only --
# the payload (exp_gene/opt_exp_gene, the reporter/expressed-gene sequence fused on
# past Kozak) is dropped before folding for these two images specifically, per this
# task's own instruction ("kozak can stay there, the payload doesn't need to be for
# the plot"). This is a real, separate fold of the shorter sequence -- not the full
# switch's real MFE structure with the tail cropped out of the picture, which would
# leave dangling, unexplained bonds wherever the full structure happened to pair into
# the payload (e.g. the outer prefix<->exp_gene stem this notebook's own mechanism
# check found). Every actual measurement elsewhere in this notebook (dG, accessibility,
# leakage, fake checks) still folds the full switch, unchanged -- only these two
# display images use the shorter sequence.
_PAYLOAD_DOMAINS = ("exp_gene", "opt_exp_gene")

for _r in single_top20:
    _switch = _r["switch"]
    _kept_domains = {name: span for name, span in _r["domains"].items() if name not in _PAYLOAD_DOMAINS}
    _display_end = max((span[1] for span in _kept_domains.values()), default=len(_switch))
    _display_seq = _switch[:_display_end]

    _mfe = folder.mfe(_display_seq)
    _matrix = folder.base_pair_probabilities(_display_seq)
    _toehold_span = _r["domains"].get("toehold")
    _spans_2d = [(_toehold_span[0], _toehold_span[1], _BINDING_COLOR_5P)] if _toehold_span else []
    _r["structure_png"] = _structure_png(
        _display_seq, _mfe.structure,
        f"{_r['csv_label']} rank {_r['rank']} — OFF (switch alone, through Kozak)",
        energy=_mfe.energy, spans=_spans_2d, folder=folder,
    )
    _r["matrix_png"] = _bpp_heatmap_png(
        _matrix, "OFF-state base-pair probabilities (through Kozak)",
        spans=[(_toehold_span[0], _toehold_span[1], _BINDING_COLOR_5P)] if _toehold_span else (),
    )
    # Text block diagram: domain name + length, in fused-frame order -- payload kept
    # here (it's a table row, not a cluttered picture, and the real switch does have
    # that domain even if the plots above don't render it).
    _ordered = sorted(_r["domains"].items(), key=lambda kv: kv[1][0])
    _r["block_diagram"] = [
        (name, span[1] - span[0]) for name, span in _ordered if span[1] > span[0]
    ]

print(f"{len(single_top20)} single-switch entries: structure + matrix images built (plots through Kozak, payload omitted)")

# ---- AND entries -- one function, called for both top-20 lists ----
def _build_and_images(entries):
    for _r in entries:
        _built = _r["built"]
        _fused = _built["fused"]
        _mfe = folder.mfe(_fused)
        _matrix = folder.base_pair_probabilities(_fused)
        _fp5, _fp3 = _built["footprint_5p"], _built["footprint_3p"]
        _r["structure_png"] = _structure_png(
            _fused, _mfe.structure,
            f"{_r['csv_label']} {_r['spacer_label']} pair {_r['rank_5p']}+{_r['rank_3p']} — OFF (no trigger)",
            energy=_mfe.energy,
            spans=[(_fp5[0], _fp5[1], _BINDING_COLOR_5P), (_fp3[0], _fp3[1], _BINDING_COLOR_3P)],
            folder=folder,
        )
        _r["matrix_png"] = _bpp_heatmap_png(
            _matrix, "OFF-state base-pair probabilities",
            spans=[(_fp5[0], _fp5[1], _BINDING_COLOR_5P), (_fp3[0], _fp3[1], _BINDING_COLOR_3P)],
        )
        _r["layout_png"] = render_layout_bar_png(_built, title="Switch layout (both hairpins, fused frame)")


_build_and_images(and_top20)
_build_and_images(and_top20_no_spacer)
_build_and_images(and_top20_spacer)
print(f"{len(and_top20)} AND entries (combined) + {len(and_top20_no_spacer)} AND entries "
      f"(no-spacer only) + {len(and_top20_spacer)} AND entries (spacer only): "
      f"structure + matrix + layout images built")

# ---- trigger-position maps: where the reported triggers sit on the sele mRNA ----
# `render_trigger_position_png` (defined in the plotting-helpers cell) takes plain
# (position, colour, label) points, so both the AND lists (two coloured roles per pair)
# and the single-switch list (one colour) feed the SAME renderer -- the tick/tier
# placement and genomic axis are not reimplemented per map.
#
# Positions come from each design's own `trigger_start`, read straight off the row that
# `load_ranked_designs` built from the CSV's trigger_start_pos column (never recomputed
# by searching the transcript for the sequence -- CLAUDE.md sec 5: a value already
# computed upstream is read, not re-derived).
def _and_trigger_points(entries):
    points = []
    for _r in entries:
        _by_rank = _and_pool(_r["csv_label"])
        _d5, _d3 = _by_rank[_r["rank_5p"]], _by_rank[_r["rank_3p"]]
        for _d, _color in ((_d5, _BINDING_COLOR_5P), (_d3, _BINDING_COLOR_3P)):
            if _d.get("trigger_start") is None:
                continue  # no position column for this export -- skipped, not faked at 0
            points.append((_d["trigger_start"], _color, str(_d["rank"])))
    return points


_AND_TRIGGER_LEGEND = (
    (_BINDING_COLOR_5P, "trigger A (5' half)"),
    (_BINDING_COLOR_3P, "trigger B (3' half)"),
)

and_trigger_map_png = render_trigger_position_png(
    _and_trigger_points(and_top20), SELE_TRANSCRIPT_LENGTH,
    f"Trigger positions on the sele mRNA ({SELE_TRANSCRIPT_LENGTH} nt) "
    f"— top {len(and_top20)} AND candidates (combined)",
    legend=_AND_TRIGGER_LEGEND,
)
and_ns_trigger_map_png = render_trigger_position_png(
    _and_trigger_points(and_top20_no_spacer), SELE_TRANSCRIPT_LENGTH,
    f"Trigger positions on the sele mRNA ({SELE_TRANSCRIPT_LENGTH} nt) "
    f"— top {len(and_top20_no_spacer)} AND candidates (no-spacer only)",
    legend=_AND_TRIGGER_LEGEND,
)
and_sp_trigger_map_png = render_trigger_position_png(
    _and_trigger_points(and_top20_spacer), SELE_TRANSCRIPT_LENGTH,
    f"Trigger positions on the sele mRNA ({SELE_TRANSCRIPT_LENGTH} nt) "
    f"— top {len(and_top20_spacer)} AND candidates (spacer only)",
    legend=_AND_TRIGGER_LEGEND,
)
single_trigger_map_png = render_trigger_position_png(
    [
        (_r["trigger_start"], _BINDING_COLOR_5P, str(_r["rank"]))
        for _r in single_top20
        if _r.get("trigger_start") is not None
    ],
    SELE_TRANSCRIPT_LENGTH,
    f"Trigger positions on the sele mRNA ({SELE_TRANSCRIPT_LENGTH} nt) "
    f"— top {len(single_top20)} single switches",
)
print("trigger-position maps built: "
      + ", ".join(
          name for name, png in (
              ("AND combined", and_trigger_map_png),
              ("AND no-spacer", and_ns_trigger_map_png),
              ("AND spacer", and_sp_trigger_map_png),
              ("single switch", single_trigger_map_png),
          ) if png is not None
      ))


In [ ]:
# Shared palette/type system for every report this notebook writes -- lives in its
# own cell (not folded into the final-report renderer) so any future report cell
# can reuse it directly, same as this notebook always has.
_MEMO_CSS = """
:root{
  --ground:#F4F2EC; --raise:#FBFAF6; --ink:#16150F; --soft:#55524A; --faint:#7C7970;
  --rule:#DFDBCD; --accent:#4A42A8; --accent-soft:#E4E1F4;
  --good:#2F6B12; --good-soft:#E0EBD2; --warn:#8A5A00; --warn-soft:#F6E9CE;
  --bad:#9E2B2B; --bad-soft:#F4DBDB;
  --sans:"IBM Plex Sans",system-ui,-apple-system,Segoe UI,sans-serif;
  --serif:"Spectral",Georgia,"Times New Roman",serif;
  --mono:"IBM Plex Mono",ui-monospace,SFMono-Regular,Menlo,monospace;
}
@media (prefers-color-scheme:dark){:root:not([data-theme="light"]){
  --ground:#14130E; --raise:#1C1B14; --ink:#F1EFE8; --soft:#B7B3A8; --faint:#8B877C;
  --rule:#302D24; --accent:#A9A2F0; --accent-soft:#252248;
  --good:#9BCB74; --good-soft:#1E2A14; --warn:#E0B060; --warn-soft:#2E2412;
  --bad:#E39292; --bad-soft:#331919;
}}
:root[data-theme="dark"]{
  --ground:#14130E; --raise:#1C1B14; --ink:#F1EFE8; --soft:#B7B3A8; --faint:#8B877C;
  --rule:#302D24; --accent:#A9A2F0; --accent-soft:#252248;
  --good:#9BCB74; --good-soft:#1E2A14; --warn:#E0B060; --warn-soft:#2E2412;
  --bad:#E39292; --bad-soft:#331919;
}
*{box-sizing:border-box}
body{margin:0;background:var(--ground);color:var(--ink);font-family:var(--sans);
  font-size:16px;line-height:1.65;-webkit-font-smoothing:antialiased;padding-inline:16px}
.wrap{max-width:860px;margin:0 auto;padding-block:44px 96px}
h1{font-family:var(--serif);font-size:clamp(28px,5vw,42px);line-height:1.12;margin:0 0 10px;
  font-weight:600;letter-spacing:-.015em;text-wrap:balance}
.dek{font-family:var(--serif);font-size:19px;font-style:italic;color:var(--soft);
  margin:0 0 28px;max-width:62ch}
.meta{font-family:var(--mono);font-size:12px;color:var(--faint);letter-spacing:.02em;
  border-top:1px solid var(--rule);border-bottom:1px solid var(--rule);
  padding-block:12px;margin-bottom:40px;display:flex;flex-wrap:wrap;gap:6px 22px}
h2{font-family:var(--serif);font-size:26px;font-weight:600;letter-spacing:-.01em;
  margin:52px 0 6px;text-wrap:balance}
h3{font-size:16px;font-weight:600;margin:28px 0 8px;letter-spacing:-.005em}
.eyebrow{font-family:var(--mono);font-size:11.5px;text-transform:uppercase;
  letter-spacing:.11em;color:var(--accent);margin:0 0 4px;font-weight:600}
p{margin:0 0 15px;max-width:68ch}
b,strong{font-weight:600}
code,.num{font-family:var(--mono);font-size:.9em;font-variant-numeric:tabular-nums}
.lede{border-left:3px solid var(--accent);padding:2px 0 2px 18px;margin:0 0 28px}
.lede p{font-size:17px}
.scroll{overflow-x:auto;margin:20px 0 26px;border:1px solid var(--rule);border-radius:3px;
  background:var(--raise)}
table{border-collapse:collapse;width:100%;font-size:13.5px;font-variant-numeric:tabular-nums}
th,td{padding:9px 13px;text-align:right;white-space:nowrap;border-bottom:1px solid var(--rule)}
th:first-child,td:first-child{text-align:left}
thead th{font-family:var(--mono);font-size:11px;text-transform:uppercase;letter-spacing:.07em;
  color:var(--faint);font-weight:600;background:var(--ground)}
tbody tr:last-child td{border-bottom:none}
tr.hi td{background:var(--good-soft)}
tr.lo td{background:var(--bad-soft)}
td.n{font-family:var(--mono)}
.callout{background:var(--accent-soft);border-radius:3px;padding:18px 20px;margin:24px 0;
  font-size:15px}
.callout p{margin:0;max-width:none}
.callout p+p{margin-top:10px}
figure.plot{margin:0;border:1px solid var(--rule);border-radius:3px;
  background:var(--raise);overflow:hidden}
figure.plot figcaption{font-family:var(--mono);font-size:11.5px;letter-spacing:.05em;
  text-transform:uppercase;color:var(--soft);padding:9px 14px;border-bottom:1px solid var(--rule)}
figure.plot figcaption b{color:var(--ink)}
.foot{margin-top:60px;padding-top:22px;border-top:1px solid var(--rule);
  font-size:13.5px;color:var(--faint)}
"""


In [ ]:
# ==================================================================================
# FINAL REPORT -- renderer. One page: top-20 single switches, top-20 AND candidates,
# both ranked by score = open - leak (never by open alone -- see the two selection
# cells above for why). Per entry: full sequence, a text block diagram (domain name +
# length) for single switches / an image block diagram for AND (both halves fused,
# from render_layout_bar_png -- already exists above), a real base-pair-probability
# heatmap, a real 2D structure plot (FoldEngine.layout_coordinates), and the fake-
# trigger statistics for that entry -- for AND, the FULL ten-state sweep
# (alone/+fake1/+fake2/+fake1+fake2/+A/+B/+A+B/+A+fake1/+B+fake2/+A+B+fake1+fake2);
# for single switches, the real NUPACK true+wrong-trigger ensemble check from the
# three-CSV comparison cell, when this candidate was one of the ones checked (some of
# the top 20 by score were not in the smaller NUPACK-checked subset -- shown as "not
# checked with a fake trigger" rather than silently left blank, per CLAUDE.md sec 3:
# never a default standing in for a measurement that was not taken).
#
# Reuses `_MEMO_CSS` (palette/type system) and the plotting helpers defined earlier in
# this notebook (`_structure_png`, `render_layout_bar_png`, `_bpp_heatmap_png`) --
# nothing here re-implements a plot. "Everything visible at once" layout (no tabs, no
# collapsed sections -- an earlier tabbed redesign of a different report in this
# notebook was explicitly reverted; the same rule applies here).
# ==================================================================================
import html as _html_final

FAKE_STATE_ORDER = (
    "alone", "+fake1", "+fake2", "+fake1+fake2",
    "+A", "+B", "+A+B", "+A+fake1", "+B+fake2", "+A+B+fake1+fake2",
)


def render_final_report_html(single_rows, and_rows, and_rows_no_spacer, and_rows_spacer_only):
    def _pct(v):
        return "--" if v is None else f"{v:.1%}"

    def _map_section(png, heading):
        """One trigger-position map, or an explicit note when the source export had no
        trigger_start_pos column to place ticks from -- never a silently missing figure."""
        if png is None:
            return (f"<h2>{heading}</h2>\n<p class=\"dim\">No trigger positions available "
                    f"for this list (the source CSV carried no trigger_start_pos).</p>")
        return (f"<h2>{heading}</h2>\n"
                f'<img src="data:image/png;base64,{png}" class="wide-img">')

    def _cb(r, key, field="check_bound_frac"):
        """Safe lookup into r["constrained"] (the hard-constrained-bind check -- may
        be absent on a row computed before that cell existed, CLAUDE.md sec 3: shown
        as "not checked", never silently defaulted to 0.0/None-as-zero)."""
        return r.get("constrained", {}).get(key, {}).get(field)

    def _spct(v):
        return "--" if v is None else f"{v:+.1%}"

    def _wrap_seq(seq, width=60):
        return "<br>".join(seq[i:i + width] for i in range(0, len(seq), width))

    # ---- summary tables (unchanged shape from the previous version) ----
    single_summary_trs = []
    for i, r in enumerate(single_rows, start=1):
        fc = r["fake_check"]
        _alone_paired = r.get("alone_real_paired_fraction")
        alone_leak = _pct(1.0 - _alone_paired) if _alone_paired is not None else "not checked"
        fake_ensemble = _pct(fc["ensemble_bound_frac"]) if fc else "not checked"
        fake_only = _pct(fc.get("fake_only_vienna_bound_frac")) if fc else "not checked"
        single_summary_trs.append(
            "<tr>"
            f"<td>{i}</td><td>{r['rank']}</td>"
            f'<td class="num">{r["trigger_len"]}</td>'
            f'<td class="num strong">{_pct(r["open"])}</td>'
            f'<td class="num leak">{_pct(r["leak"])}</td>'
            f'<td class="num leak">{alone_leak}</td>'
            f'<td class="num">{fake_ensemble}</td>'
            f'<td class="num leak">{fake_only}</td>'
            f'<td class="num">{_spct(r["score"])}</td>'
            f'<td class="num">{r["d_ee"]:.1f}</td>'
            f'<td><a class="btn" href="#single-{i}">plots &amp; detail &darr;</a></td>'
            "</tr>"
        )

    def _and_summary_rows(rows, id_prefix):
        trs = []
        for i, r in enumerate(rows, start=1):
            st = r["states"]
            trs.append(
                "<tr>"
                f"<td>{i}</td>"
                f"<td>{_html_final.escape(r['spacer_label'])}</td>"
                f"<td>{r['rank_5p']}+{r['rank_3p']}</td>"
                f'<td class="num strong">{_pct(r["open"])}</td>'
                f'<td class="num leak">{_pct(r["leak"])}</td>'
                f'<td class="num leak">{_pct(st["+A"]["sampled_both"])}</td>'
                f'<td class="num leak">{_pct(st["+B"]["sampled_both"])}</td>'
                f'<td class="num leak">{_pct(st["+A+fake1"]["sampled_both"])}</td>'
                f'<td class="num leak">{_pct(st["+B+fake2"]["sampled_both"])}</td>'
                f'<td class="num">{_pct(_cb(r, "A_bound_check_B"))}</td>'
                f'<td class="num">{_pct(_cb(r, "B_bound_check_A"))}</td>'
                f'<td class="num">{_pct(_cb(r, "A_bound_check_B_fake"))}</td>'
                f'<td class="num">{_pct(_cb(r, "B_bound_check_A_fake"))}</td>'
                f'<td class="num leak">{_pct(_cb(r, "A_bound_check_fakeonly"))}</td>'
                f'<td class="num leak">{_pct(_cb(r, "B_bound_check_fakeonly"))}</td>'
                f'<td class="num">{_spct(r["score"])}</td>'
                f'<td class="num">{r["d_ee_worst_alt"]:.1f}</td>'
                f'<td><a class="btn" href="#{id_prefix}-{i}">plots &amp; detail &darr;</a></td>'
                "</tr>"
            )
        return "".join(trs)

    and_summary_trs = _and_summary_rows(and_rows, "and")
    and_ns_summary_trs = _and_summary_rows(and_rows_no_spacer, "and-ns")
    and_sp_summary_trs = _and_summary_rows(and_rows_spacer_only, "and-sp")

    # ---- single-switch detail cards ----
    single_cards = []
    for i, r in enumerate(single_rows, start=1):
        block_rows = "".join(
            f"<tr><td>{_html_final.escape(name)}</td><td class='num'>{length} nt</td></tr>"
            for name, length in r["block_diagram"]
        )
        fc = r["fake_check"]
        if fc is None:
            fake_html = '<p class="dim">Not in the NUPACK-checked subset for this CSV -- no real true+wrong-trigger ensemble result for this specific candidate.</p>'
        else:
            _alone_paired = r.get("alone_real_paired_fraction")
            _alone_row = (
                f'<tr><td><b>Toehold ALONE (REAL Boltzmann sampling, no trigger/fake present) -- '
                f'P(footprint paired -- only possible partner is its own stem_down)</b></td>'
                f'<td class="num strong">{_pct(_alone_paired)}</td></tr>'
                if _alone_paired is not None else
                '<tr><td>Toehold ALONE, real Boltzmann sampling</td><td class="num">not checked</td></tr>'
            )
            _coverage_line = (
                f'Sampling coverage (clean ON check): {_pct(r.get("sample_coverage"))} of the '
                f'effective ensemble (~{r["effective_structures"]:.1f} distinct structures carry '
                f'the bulk of the real Boltzmann probability mass for this molecule).'
                if r.get("sample_coverage") is not None
                else "Sampling coverage: not available for this candidate."
            )
            fake_html = f"""
            <table>
              <thead><tr><th>metric</th><th>value</th></tr></thead>
              <tbody>
                {_alone_row}
                <tr><td>OFF-alone leakage (NUPACK tube -- degenerate, see notebook note)</td><td class="num leak">{_pct(fc['leakage_frac'])}</td></tr>
                <tr><td>ON, true trigger only (NUPACK tube, correct complex)</td><td class="num">{_pct(fc['on_correct_frac'])}</td></tr>
                <tr><td>[tube+MFE only] true+wrong, correct complex</td><td class="num">{_pct(fc['cross_correct_frac_tube_mfe_only'])}</td></tr>
                <tr><td>[tube+MFE only] MFE intended-pair fraction</td><td class="num">{_pct(fc['mfe_intended_frac_tube_mfe_only'])}</td></tr>
                <tr><td><b>[REAL Boltzmann-sampled ensemble] true trigger bound, wrong trigger also present</b></td>
                    <td class="num strong">{_pct(fc['ensemble_bound_frac'])}</td></tr>
                <tr><td>FAKE ONLY (true trigger absent), NUPACK tube -- spurious binding to the wrong trigger alone</td>
                    <td class="num leak">{_pct(fc.get('fake_only_nupack_bound_frac'))}</td></tr>
                <tr><td><b>FAKE ONLY (true trigger absent), REAL Boltzmann sampling -- footprint closes over the wrong trigger alone</b></td>
                    <td class="num leak">{_pct(fc.get('fake_only_vienna_bound_frac'))}</td></tr>
              </tbody>
            </table>
            <p class="dim">{fc['ensemble_n_samples']} independently sampled structures. {_coverage_line}</p>"""
        single_cards.append(f"""
        <div class="card" id="single-{i}">
          <h3>#{i} — rank {r['rank']}
            <span class="dim">score {_spct(r['score'])} (open {_pct(r['open'])}, leak {_pct(r['leak'])})</span>
          </h3>
          <p class="seq"><b>Switch sequence</b> ({len(r['switch'])} nt):<br>{_wrap_seq(r['switch'])}</p>
          <p class="seq"><b>Trigger</b> ({len(r['trigger'])} nt): {r['trigger']}</p>
          <h4>Block diagram (domain — length)</h4>
          <table class="block-table"><tbody>{block_rows}</tbody></table>
          <div class="imgrow">
            <figure><img src="data:image/png;base64,{r['structure_png']}"><figcaption>2D structure (OFF)</figcaption></figure>
            <figure><img src="data:image/png;base64,{r['matrix_png']}"><figcaption>Base-pair probability matrix (OFF)</figcaption></figure>
          </div>
          <h4>Fake-trigger check (true trigger + one wrong trigger, real ensemble)</h4>
          {fake_html}
        </div>""")

    # ---- AND detail cards -- one function, called for both top-20 lists ----
    def _and_card_html(r, i, id_prefix):
        built = r["built"]
        _alone_cov = r["states"].get("alone", {})
        _and_coverage_line = (
            f'Sampling coverage (alone state): {_pct(_alone_cov.get("sample_coverage"))} of the '
            f'effective ensemble (~{_alone_cov["effective_structures"]:.1f} distinct structures '
            f'carry the bulk of the real Boltzmann probability mass for this molecule).'
            if _alone_cov.get("sample_coverage") is not None
            else "Sampling coverage: not available for this pair."
        )
        _spacer_seq = r.get("spacer_seq")
        _spacer_line = (
            f'<p class="seq"><b>Spacer</b> ({len(_spacer_seq)} nt, optimized for this pair -- '
            f'worst complementary run to either toehold/trigger: {r.get("spacer_worst_run")}nt): '
            f'{_spacer_seq}</p>'
            if _spacer_seq else ""
        )
        state_rows = []
        for name in FAKE_STATE_ORDER:
            st = r["states"][name]
            state_rows.append(
                "<tr>"
                f"<td>{_html_final.escape(name)}</td>"
                f'<td class="num">{_pct(st["sampled_a_open"])}</td>'
                f'<td class="num">{_pct(st["sampled_b_open"])}</td>'
                f'<td class="num strong">{_pct(st["sampled_both"])}</td>'
                "</tr>"
            )
        _constrained = r.get("constrained")
        if _constrained is None:
            _constrained_html = '<p class="dim">Not checked for this pair (computed by a cell added after this run).</p>'
        else:
            _ab = _constrained["A_bound_check_B"]
            _ba = _constrained["B_bound_check_A"]
            _abf = _constrained["A_bound_check_B_fake"]
            _baf = _constrained["B_bound_check_A_fake"]
            _abfo = _constrained["A_bound_check_fakeonly"]
            _bafo = _constrained["B_bound_check_fakeonly"]
            _constrained_html = f"""
            <table>
              <thead><tr><th>forced</th><th>checked</th><th>P(checked bound)</th><th>n forced pairs / footprint len</th></tr></thead>
              <tbody>
                <tr><td>A bound to trigger A (real duplex)</td><td>B, alone</td>
                    <td class="num strong">{_pct(_ab["check_bound_frac"])}</td>
                    <td class="num">{_ab["n_forced_pairs"]}/{_ab["footprint_bound_len"]}</td></tr>
                <tr><td>B bound to trigger B (real duplex)</td><td>A, alone</td>
                    <td class="num strong">{_pct(_ba["check_bound_frac"])}</td>
                    <td class="num">{_ba["n_forced_pairs"]}/{_ba["footprint_bound_len"]}</td></tr>
                <tr><td>A bound to trigger A (real duplex)</td><td>B, + 1 off-target fake</td>
                    <td class="num strong">{_pct(_abf["check_bound_frac"])}</td>
                    <td class="num">{_abf["n_forced_pairs"]}/{_ab["footprint_bound_len"]} (mean of {_abf["n_fakes_checked"]} fakes)</td></tr>
                <tr><td>B bound to trigger B (real duplex)</td><td>A, + 1 off-target fake</td>
                    <td class="num strong">{_pct(_baf["check_bound_frac"])}</td>
                    <td class="num">{_baf["n_forced_pairs"]}/{_ba["footprint_bound_len"]} (mean of {_baf["n_fakes_checked"]} fakes)</td></tr>
                <tr><td>A bound to trigger A (real duplex)</td><td>off-target fake ONLY (B absent)</td>
                    <td class="num leak">{_pct(_abfo["check_bound_frac"])}</td>
                    <td class="num">{_abfo["n_forced_pairs"]}/{_ab["footprint_bound_len"]} (mean of {_abfo["n_fakes_checked"]} fakes)</td></tr>
                <tr><td>B bound to trigger B (real duplex)</td><td>off-target fake ONLY (A absent)</td>
                    <td class="num leak">{_pct(_bafo["check_bound_frac"])}</td>
                    <td class="num">{_bafo["n_forced_pairs"]}/{_ba["footprint_bound_len"]} (mean of {_bafo["n_fakes_checked"]} fakes)</td></tr>
              </tbody>
            </table>
            <p class="dim">Conditional check, not the real unconstrained joint probability above: one role is
            HARD-CONSTRAINED to its own real bound duplex (read off an actual MFE fold), then the other
            role's real binding is measured in what remains. Answers "CAN the other role still bind GIVEN
            this one is genuinely bound", not "does the real system actually choose both on its own" --
            that question is what the ten-state table above already answers via sampled_both.</p>"""

        return f"""
        <div class="card" id="{id_prefix}-{i}">
          <h3>#{i} — {_html_final.escape(r['spacer_label'])}, pair {r['rank_5p']}+{r['rank_3p']}
            <span class="dim">score {_spct(r['score'])} (open {_pct(r['open'])}, leak {_pct(r['leak'])})</span>
          </h3>
          <p class="seq"><b>Fused switch sequence</b> ({len(built['fused'])} nt):<br>{_wrap_seq(built['fused'])}</p>
          <p class="seq"><b>Trigger A</b> ({len(r['trigger_a'])} nt): {r['trigger_a']}</p>
          <p class="seq"><b>Trigger B</b> ({len(r['trigger_b'])} nt): {r['trigger_b']}</p>
          {_spacer_line}
          <h4>Block diagram (both hairpins, fused frame)</h4>
          <img src="data:image/png;base64,{r['layout_png']}" class="wide-img">
          <div class="imgrow">
            <figure><img src="data:image/png;base64,{r['structure_png']}"><figcaption>2D structure (OFF)</figcaption></figure>
            <figure><img src="data:image/png;base64,{r['matrix_png']}"><figcaption>Base-pair probability matrix (OFF)</figcaption></figure>
          </div>
          <h4>Fake-trigger sweep — all ten states, real Boltzmann sampling</h4>
          <table>
            <thead><tr><th>state</th><th>A open</th><th>B open</th><th>BOTH (joint)</th></tr></thead>
            <tbody>{"".join(state_rows)}</tbody>
          </table>
          <p class="dim">{_and_coverage_line}</p>
          <h4>Constrained-bind check — one role forced to its real bound duplex, other role checked</h4>
          {_constrained_html}
        </div>"""

    and_cards = [_and_card_html(r, i, "and") for i, r in enumerate(and_rows, start=1)]
    and_ns_cards = [_and_card_html(r, i, "and-ns") for i, r in enumerate(and_rows_no_spacer, start=1)]
    and_sp_cards = [_and_card_html(r, i, "and-sp") for i, r in enumerate(and_rows_spacer_only, start=1)]

    body = f"""<!doctype html>
<html><head><meta charset="utf-8">
<title>Toehold switch — final ranked report</title>
<style>{_MEMO_CSS}
table{{width:100%;border-collapse:collapse;font-family:var(--mono);font-size:12.5px;margin:10px 0 20px}}
th,td{{border-bottom:1px solid var(--rule);padding:6px 9px;text-align:right}}
th{{color:var(--faint);font-weight:600;font-size:10.5px;vertical-align:bottom}}
td:first-child,th:first-child,td:nth-child(2),th:nth-child(2){{text-align:left}}
.num.strong{{color:var(--accent);font-weight:700}}
.num.leak{{color:var(--warn)}}
.card{{border:1px solid var(--rule);border-radius:4px;padding:18px 20px;margin:0 0 24px;background:var(--raise)}}
.card h3{{margin-top:0}}
.card h4{{font-size:13px;margin:18px 0 6px}}
.seq{{font-family:var(--mono);font-size:11.5px;word-break:break-all;line-height:1.5}}
.block-table{{max-width:360px}}
.imgrow{{display:flex;flex-wrap:wrap;gap:14px;margin:10px 0}}
.imgrow figure{{margin:0}}
.imgrow img{{max-width:340px;border:1px solid var(--rule);border-radius:3px}}
.imgrow figcaption{{font-size:11px;color:var(--faint);text-align:center;margin-top:4px}}
.wide-img{{max-width:100%;border:1px solid var(--rule);border-radius:3px;margin:8px 0}}
.btn{{font-size:11.5px;color:var(--accent);text-decoration:none;white-space:nowrap}}
.btn:hover{{text-decoration:underline}}
</style></head>
<body><div class="wrap">
<h1>Final ranked report</h1>
<p class="dek">Top 20 single-switch designs and top 20 AND candidates, each ranked by
real Boltzmann-sampled signal minus its own real leakage — never by signal alone.</p>
<div class="meta"><span>{TEMPERATURE_C:.1f}&deg;C</span>
<span>{len(THREE_CSV_LABELS)} source CSV{"s" if len(THREE_CSV_LABELS) != 1 else ""}</span><span>score = open - leak</span></div>

{_map_section(single_trigger_map_png, "Trigger positions on the gene — top 20 single switch")}

<h2>Summary — top 20 single switch</h2>
<p class="headline">"alone leak" = 1 - real Boltzmann-sampled P(footprint paired) with
NOTHING present at all -- no trigger, no fake. With nothing else in the tube, the only
partner the footprint could pair to is its own stem_down, so this is a genuine
spontaneous-opening measurement, not the NUPACK tube number, which is structurally
near-zero regardless of sequence (single-strand tube has nowhere else to partition to).
"fake ensemble" = real Boltzmann-sampled P(true trigger still
bound, one wrong trigger also present). "fake only" = real Boltzmann-sampled
P(footprint closes over the WRONG trigger, true trigger absent entirely) -- ideally
near 0%, since an off-target should not complement this switch's own toehold at all.
All three are the MEAN over up to 10 distinct, non-overlapping fake triggers per
candidate where they depend on a fake choice at all (alone leak does not).</p>
<table>
<thead><tr><th>#</th><th>rank</th><th>trig nt</th><th>open</th>
<th>leak</th><th>alone leak</th><th>fake ensemble</th><th>fake only</th><th>score</th><th>dG (kcal/mol)</th><th></th></tr></thead>
<tbody>{"".join(single_summary_trs)}</tbody>
</table>

{_map_section(and_trigger_map_png, "Trigger positions on the gene — top 20 AND (combined)")}

<h2>Summary — top 20 AND</h2>
<p class="headline">"open" = min(+A+B both, +A+B+fake1+fake2 both) -- the worse of
"does giving it both real triggers activate it" and "does that survive two off-targets
also being present". "leak (worst)" = the WORST joint-both-open reading across every
state that should NOT look fully activated: alone, +fake1, +fake2, +fake1+fake2, +A
alone, +B alone, +A+fake1, +B+fake2 -- this is what the ranking itself is built from,
not just a display number. "+A both"/"+B both" = real joint P(both footprints open)
with only ONE real trigger present and nothing else -- the direct test of whether a
single input alone already drives a false "AND satisfied" reading (the OR-instead-of-
AND failure mode). "+A+fake1"/"+B+fake2" = the same false-activation question with an
off-target fake added to the single real trigger. All four single-state columns are
the MEAN over up to 10 distinct, non-overlapping fake-trigger draws per pair where they
depend on a fake choice at all (+A/+B do not).</p>
<table>
<thead><tr><th>#</th><th>spacer</th><th>pair</th><th>open (joint)</th>
<th>leak (worst)</th><th>+A both</th><th>+B both</th><th>+A+fake1</th><th>+B+fake2</th><th>A bound&rarr;B</th><th>B bound&rarr;A</th><th>A bound&rarr;B+fake</th><th>B bound&rarr;A+fake</th><th>A bound&rarr;fakeonly</th><th>B bound&rarr;fakeonly</th><th>score</th><th>dG worst-alt</th><th></th></tr></thead>
<tbody>{and_summary_trs}</tbody>
</table>

{_map_section(and_ns_trigger_map_png, "Trigger positions on the gene — top 20 AND (no-spacer only)")}

<h2>Summary — top 20 AND, no-spacer variant only</h2>
<p class="headline">Same ranking and same score formula as the combined AND table
above, but the pool is restricted to no-spacer pairs before ranking -- the combined
table above never shows a no-spacer pair because the spacer variant's scores are
higher across the board, so this table is what the no-spacer construct's own best
candidates actually look like on their own terms.</p>
<table>
<thead><tr><th>#</th><th>spacer</th><th>pair</th><th>open (joint)</th>
<th>leak (worst)</th><th>+A both</th><th>+B both</th><th>+A+fake1</th><th>+B+fake2</th><th>A bound&rarr;B</th><th>B bound&rarr;A</th><th>A bound&rarr;B+fake</th><th>B bound&rarr;A+fake</th><th>A bound&rarr;fakeonly</th><th>B bound&rarr;fakeonly</th><th>score</th><th>dG worst-alt</th><th></th></tr></thead>
<tbody>{and_ns_summary_trs}</tbody>
</table>

{_map_section(and_sp_trigger_map_png, "Trigger positions on the gene — top 20 AND (spacer only)")}

<h2>Summary — top 20 AND, spacer variant only</h2>
<p class="headline">Same ranking and same score formula again, this time the pool
restricted to spacer pairs before ranking -- the mirror of the no-spacer-only table
above, so the spacer construct's own best candidates are visible on their own terms
too, not just as the pairs that happen to fill the combined table.</p>
<table>
<thead><tr><th>#</th><th>spacer</th><th>pair</th><th>open (joint)</th>
<th>leak (worst)</th><th>+A both</th><th>+B both</th><th>+A+fake1</th><th>+B+fake2</th><th>A bound&rarr;B</th><th>B bound&rarr;A</th><th>A bound&rarr;B+fake</th><th>B bound&rarr;A+fake</th><th>A bound&rarr;fakeonly</th><th>B bound&rarr;fakeonly</th><th>score</th><th>dG worst-alt</th><th></th></tr></thead>
<tbody>{and_sp_summary_trs}</tbody>
</table>

<h2>Detail — single switch, sequence + structure + fake-trigger check</h2>
{"".join(single_cards)}

<h2>Detail — AND, sequence + structure + full ten-state fake sweep</h2>
{"".join(and_cards)}

<h2>Detail — AND (no-spacer variant), sequence + structure + full ten-state fake sweep</h2>
{"".join(and_ns_cards)}

<h2>Detail — AND (spacer variant), sequence + structure + full ten-state fake sweep</h2>
{"".join(and_sp_cards)}

<p class="dim" style="margin-top:40px">Generated by render_final_report_html().</p>
</div></body></html>"""
    return body


In [ ]:
# ===== FINAL REPORT: write the HTML =====
import datetime as _dt_final

_final_report_root = Path.cwd()
for _base in (_final_report_root, *_final_report_root.parents):
    if (_base / "pyproject.toml").exists():
        _final_report_root = _base / "var" / "reports" / f"{_dt_final.date.today().isoformat()}_final-ranked-report"
        break

_final_report_root.mkdir(parents=True, exist_ok=True)
final_report_path = _final_report_root / "final_ranked_report.html"
final_report_path.write_text(render_final_report_html(single_top20, and_top20, and_top20_no_spacer, and_top20_spacer))
print(f"wrote {final_report_path.stat().st_size:,} bytes to {final_report_path}")


In [ ]:
# ===== FINAL REPORT: render the PDF =====
import subprocess as _subprocess_final
import sys as _sys_final

_final_render_pdf_script = None
for _base in (Path.cwd(), *Path.cwd().parents):
    _candidate = _base / "src/engine/gates/notebooks/toehold/render_pdf.py"
    if _candidate.exists():
        _final_render_pdf_script = _candidate
        break

if _final_render_pdf_script is None:
    print("render_pdf.py not found -- HTML written, PDF skipped")
else:
    final_report_pdf_path = final_report_path.with_suffix(".pdf")
    _subprocess_final.run(
        [_sys_final.executable, str(_final_render_pdf_script),
         str(final_report_path), str(final_report_pdf_path)],
        check=True,
    )
    print(f"wrote {final_report_pdf_path}")


In [ ]:
# ===== FINAL REPORT: ready to publish =====
# This is the deliverable: `final_report_path` (HTML) is what gets handed to the
# Artifact tool to publish/update the live report page. A notebook cell cannot call
# that tool itself -- this cell only confirms the file is in place and prints exactly
# what to publish.
print("FINAL REPORT ready to publish:")
print(f"  HTML: {final_report_path}")
print(f"  PDF:  {final_report_pdf_path if _final_render_pdf_script else '(not rendered)'}")
print(f"  {len(single_top20)} single-switch candidates, {len(and_top20)} AND candidates "
      f"(combined) + {len(and_top20_no_spacer)} AND candidates (no-spacer only) + "
      f"{len(and_top20_spacer)} AND candidates (spacer only), "
      f"both ranked by real signal minus real leakage.")


In [ ]:
# ==================================================================================
# WHY DOES THE SPACER VARIANT LEAK MORE? -- a real, data-grounded mechanism check, not
# a guess. `leak` = max(alone A open, alone B open): the worse-behaved footprint's own
# real Boltzmann-sampled open fraction with NEITHER trigger present. The question is
# why inserting SPACER_SEQ between the two hairpins raises that number for the SAME
# rank pair, using the SAME two designs -- the only thing that changed is the spacer.
#
# Method: pick the highest-leak entry in the combined AND top-20, pull the SAME
# (csv, rank_5p, rank_3p) pair's "alone" state from BOTH mega_sweep_results configs
# (spacer / no_spacer -- both already computed by the mega sweep, no new folding), then
# fold both actual fused molecules (fuse() with spacer=SPACER_SEQ vs spacer="") and
# read off, base by base, what the leaky footprint pairs with in each MFE structure.
# ==================================================================================
_leak_example = max(and_top20, key=lambda r: r["leak"])
_le_csv, _le_r5, _le_r3 = _leak_example["csv_label"], _leak_example["rank_5p"], _leak_example["rank_3p"]
print(f"Highest-leak entry in the combined AND top-20: {_le_csv}, pair {_le_r5}+{_le_r3}, "
      f"open {_leak_example['open']:.1%}, leak {_leak_example['leak']:.1%}")

# ---- 1. same pair, both spacer configs, straight from the already-computed mega sweep ----
for _spacer_label in ("no_spacer", "spacer"):
    _rows = mega_sweep_results[(_le_csv, _spacer_label)]
    _match = next((r for r in _rows if r["rank_5p"] == _le_r5 and r["rank_3p"] == _le_r3), None)
    if _match is None:
        print(f"  [{_spacer_label:>9}] pair {_le_r5}+{_le_r3} was excluded from this sweep "
              f"(overlap or no fake available)")
        continue
    _alone = _match["states"]["alone"]
    print(f"  [{_spacer_label:>9}] alone: A open {_alone['sampled_a_open']:.1%}  "
          f"B open {_alone['sampled_b_open']:.1%}  BOTH {_alone['sampled_both']:.1%}")

# ---- 2. build both actual molecules and look at the MFE structure directly ----
_le_pool = _and_pool(_le_csv)
_le_d5, _le_d3 = _le_pool[_le_r5], _le_pool[_le_r3]
_le_built_spacer, _le_problem_s = fuse(_le_d5, _le_d3, spacer=SPACER_SEQ)
_le_built_plain, _le_problem_p = fuse(_le_d5, _le_d3, spacer="")
assert _le_problem_s is None and _le_problem_p is None

_le_mfe_spacer = folder.mfe(_le_built_spacer["fused"])
_le_mfe_plain = folder.mfe(_le_built_plain["fused"])


def _pairs_touching(structure, span):
    """(position, partner_or_None) for every position in [span[0], span[1]), from a
    dot-bracket string -- a direct, position-by-position read of what each footprint
    base is actually doing in the MFE structure, not a summary statistic."""
    n = len(structure)
    stack = []
    partner = [None] * n
    for i, ch in enumerate(structure):
        if ch == "(":
            stack.append(i)
        elif ch == ")":
            j = stack.pop()
            partner[i] = j
            partner[j] = i
    return [(p, partner[p]) for p in range(span[0], span[1])]


# Whichever footprint (5' = A, 3' = B) is the leaky one for this pair, per the real
# sampled fractions just printed above.
_leaky_side = "5p" if _leak_example["states"]["alone"]["sampled_a_open"] >= _leak_example["states"]["alone"]["sampled_b_open"] else "3p"
_fp_plain = _le_built_plain[f"footprint_{_leaky_side}"]
_fp_spacer = _le_built_spacer[f"footprint_{_leaky_side}"]
print(f"\nLeaky side: {_leaky_side} -- footprint (no spacer) {_fp_plain}, footprint (spacer) {_fp_spacer}")
print(f"Spacer span in the spacer-built molecule: {_le_built_spacer['spacer_span']}")

print(f"\n[no spacer] MFE energy {_le_mfe_plain.energy:.1f} kcal/mol -- footprint base-pairing:")
for pos, partner in _pairs_touching(_le_mfe_plain.structure, _fp_plain):
    where = "unpaired" if partner is None else (
        "-> inside its own footprint" if _fp_plain[0] <= partner < _fp_plain[1]
        else f"-> position {partner} (outside its own footprint)"
    )
    print(f"    pos {pos}: {where}")

print(f"\n[spacer]    MFE energy {_le_mfe_spacer.energy:.1f} kcal/mol -- footprint base-pairing:")
for pos, partner in _pairs_touching(_le_mfe_spacer.structure, _fp_spacer):
    where = "unpaired" if partner is None else (
        "-> inside its own footprint" if _fp_spacer[0] <= partner < _fp_spacer[1]
        else f"-> position {partner} (outside its own footprint)"
    )
    print(f"    pos {pos}: {where}")

In [ ]:
# ==================================================================================
# A/B ORDER SWAP -- does the "2nd" role open while the "1st" stays stuck, regardless
# of WHICH design is in it? The first test (sele_old, pair 130+7) showed the 5' role
# failing for BOTH designs when each took that role in turn -- but that is only
# evidence for THIS ONE PAIR. If it were sequence-specific complementarity (something
# about one design's freed sequence happening to interact with the other's footprint),
# the pattern would depend on WHICH two designs are paired, not on which slot they
# sit in -- and a single pair testing "does 5' fail" in both directions cannot tell
# a real universal positional cause apart from a coincidence specific to these two
# sequences. The only way to tell them apart: run the SAME swap test on a second,
# unrelated pair (different CSV, different designs, zero rank overlap with the first)
# and see whether 5' fails there too.
#
# One function, called on two independent pairs -- not two copies of the test.
# ==================================================================================
import sys as _sys_swap
from pathlib import Path as _Path_swap

_here_swap = _Path_swap.cwd()
for _base in (_here_swap, *_here_swap.parents):
    if (_base / "_joint_state_parallel.py").exists():
        if str(_base) not in _sys_swap.path:
            _sys_swap.path.insert(0, str(_base))
        break
from _joint_state_parallel import FakeSweepJob, pick_fake_design, run_parallel_fake_sweep


def run_swap_test(csv_path, rank_a, rank_b, spacer=""):
    """fuse()s the SAME two designs in both (5',3') orders and runs the real
    +A/+B/+A+B check on both -- returns nothing, just prints, since this is a
    diagnostic, not a value other cells build on."""
    pool, _ = load_ranked_designs(csv_path, folder, limit=150)
    by_rank = {d["rank"]: d for d in pool}
    d_a = by_rank.get(str(rank_a), by_rank.get(rank_a))
    d_b = by_rank.get(str(rank_b), by_rank.get(rank_b))
    assert d_a is not None and d_b is not None, f"rank {rank_a} or {rank_b} not found in {csv_path}"

    jobs = []
    built_by_label = {}
    for order_label, d5, d3 in (
        (f"published order ({rank_a}=5p, {rank_b}=3p)", d_a, d_b),
        (f"SWAPPED ({rank_b}=5p, {rank_a}=3p)", d_b, d_a),
    ):
        built, problem = fuse(d5, d3, spacer=spacer)
        assert problem is None, problem
        fake1 = pick_fake_design(d5, d3, {d5["rank"], d3["rank"]}, pool, start_offset=0)
        fake2 = pick_fake_design(d5, d3, {d5["rank"], d3["rank"], fake1["rank"]}, pool, start_offset=1)
        built_by_label[order_label] = (d5, d3, built)
        jobs.append(FakeSweepJob(
            key=order_label, fused=built["fused"], trigger_a=d5["trigger"], trigger_b=d3["trigger"],
            fake_a=fake1["trigger"], fake_b=fake2["trigger"],
            footprint_5p=built["footprint_5p"], footprint_3p=built["footprint_3p"],
            temperature=TEMPERATURE_C, n_samples=1200, threshold=0.80,
        ))

    results = run_parallel_fake_sweep(jobs, max_workers=2)

    print(f"Designs: rank {rank_a} (own toehold {d_a['domains']['toehold']}) and "
          f"rank {rank_b} (own toehold {d_b['domains']['toehold']}) -- {csv_path}")
    print()
    for order_label, (d5, d3, built) in built_by_label.items():
        res = results[order_label]
        print(f"=== {order_label} -- 5' = rank {d5['rank']}, 3' = rank {d3['rank']} ===")
        for state in ("+A", "+B", "+A+B"):
            st = res[state]
            print(f"  {state:>6}: A(5p) open {st['sampled_a_open']:.1%}  "
                  f"B(3p) open {st['sampled_b_open']:.1%}  BOTH {st['sampled_both']:.1%}")
        print()
    return built_by_label, results


print("##### PAIR 1: sele_old, ranks 130+7 (the original test) #####")
# kept at module level (_swap_built/_swap_results) -- the mechanism-tracing cells
# right after this one dig into PAIR 1's actual MFE structure and need these.
_swap_built, _swap_results = run_swap_test(
    "/Users/zeev/Projects/TOEHOLDS/sele_old/basic_switch/results_ranked.csv", 130, 7
)

print("##### PAIR 2: sele, ranks 86+49 (independent -- different CSV, different designs, no rank overlap) #####")
run_swap_test("/Users/zeev/Projects/TOEHOLDS/sele/basic_switch/results_ranked.csv", 86, 49)


In [ ]:
# ==================================================================================
# WHY DOES THE 5' ROLE STAY LOCKED? -- real MFE structure of the "alone" (no trigger)
# fused molecule, published order (130=5', 7=3'), reading off exactly what the 5'
# footprint (rank 130's toehold+stem_up) pairs to. Same base-pair-partner tracing this
# notebook already used to find the spacer-complementarity mechanism earlier.
# ==================================================================================
_lock_built = _swap_built["published order (130=5p, 7=3p)"][2]
_lock_fused = _lock_built["fused"]
_lock_mfe = folder.mfe(_lock_fused)
_fp5, _fp3 = _lock_built["footprint_5p"], _lock_built["footprint_3p"]


def _pairs_touching(structure, span):
    n = len(structure)
    stack, partner = [], [None] * n
    for i, ch in enumerate(structure):
        if ch == "(":
            stack.append(i)
        elif ch == ")":
            j = stack.pop()
            partner[i] = j
            partner[j] = i
    return [(p, partner[p]) for p in range(span[0], span[1])]


print(f"fused length {len(_lock_fused)}, MFE energy {_lock_mfe.energy:.1f} kcal/mol")
print(f"footprint_5p (rank 130, the locked role) = {_fp5}")
print(f"footprint_3p (rank 7, the working role) = {_fp3}\n")

print("5' footprint (locked) base-pairing in the alone-state MFE structure:")
for pos, partner in _pairs_touching(_lock_mfe.structure, _fp5):
    where = "unpaired" if partner is None else (
        "-> inside its own 5p footprint" if _fp5[0] <= partner < _fp5[1]
        else f"-> position {partner} (outside 5p footprint)"
    )
    print(f"    pos {pos}: {where}")

print("\n3' footprint (working) base-pairing in the alone-state MFE structure:")
for pos, partner in _pairs_touching(_lock_mfe.structure, _fp3):
    where = "unpaired" if partner is None else (
        "-> inside its own 3p footprint" if _fp3[0] <= partner < _fp3[1]
        else f"-> position {partner} (outside 3p footprint)"
    )
    print(f"    pos {pos}: {where}")


In [ ]:
# ---- follow-up: why does +A alone open (100%) but +A+B does not (0%)? Fold the real
# three-strand +A+B complex and read off what the 5' footprint pairs to THERE.
_lock_d5, _lock_d3 = _swap_built["published order (130=5p, 7=3p)"][:2]
_lock_ab = f"{_lock_fused}&{_lock_d5['trigger']}&{_lock_d3['trigger']}"
_lock_ab_mfe = folder.mfe(_lock_ab)
print(f"+A+B complex length {len(_lock_ab)}, MFE energy {_lock_ab_mfe.energy:.1f} kcal/mol")
print()
print("5' footprint (locked, even with both triggers present) base-pairing in +A+B MFE:")
for pos, partner in _pairs_touching(_lock_ab_mfe.structure, _fp5):
    if partner is None:
        where = "unpaired"
    elif _fp5[0] <= partner < _fp5[1]:
        where = "-> inside its own 5p footprint"
    elif partner < len(_lock_fused):
        where = f"-> position {partner} (elsewhere in the fused switch)"
    else:
        _trig_a_start = len(_lock_fused) + 1
        _trig_b_start = _trig_a_start + len(_lock_d5["trigger"]) + 1
        which_trig = "trigger A" if partner < _trig_b_start else "trigger B"
        where = f"-> position {partner} ({which_trig})"
    print(f"    pos {pos}: {where}")
